# GwenLand glcuda - T4 Ceiling Wave 13B

**One GEMM for Q, K and V.**

Wave 13A lit up the stages that reported nothing and left `qkv` as the worst
GEMM in the engine: 1249 GMAC/s against `ffn_gate_up`'s 4236. The reason is
launch geometry, not arithmetic. `k` and `v` are 128-row projections, so each
one fills **eight CTAs** on a 40-SM T4 and does that twice per layer, 48 times
per prefill. Stacked with `q` they are one launch of 72 CTAs.

The isolated screening measured it: three launches 123.54 us against one stacked
launch 58.00 us, **2.13x**, bit-exact. `qkv` is 8.6% of prefill, so the honest
end-to-end expectation is roughly **+4 to +6%** - which straddles the repo's 5%
retention bar. Both decision thresholds are therefore fixed in the notebook
before the run, not after.

Stacking the weights is free (row-major SoA concatenates along rows). The cost
is that Q/K/V become column slices of one slab, so five kernels had to learn a
row stride they never had. Exactly which PTX entries changed is a hard gate
here, as is a new parity test that runs the whole bias -> RoPE -> KV-write ->
attention chain over a strided slice and demands the packed result bit for bit.

Design: `architecture/glcuda-research/ceiling-sprint-wave13b-qkv-stacking.md`.


## 1 - Bootstrap, patch stack, and the stride-migration contract

In [ ]:
import ast
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time
import traceback
import urllib.error
import urllib.request
import zipfile

NOTEBOOK_BUILD = "wave13b-qkv-stacking-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE11_PATCH_SHA256 = "0e644ae9c37ecdf73fd354a8c59100ed31bb3edce5c76e13b5f08ef4dd3292ca"
WAVE11_PATCH_GZIP_B64 = """..."""
WAVE12_PATCH_SHA256 = "9920afe7f160cebcb51ae0d0dfeca41ebfaa8532285e1efcb92245cd8fd2fee0"
WAVE13B_PATCH_SHA256 = "51aed3b9a31adb8d902257c605b79181bc1973ef5ab123a5036148a0e6ca878c"
WAVE13B_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19/XfbNrLo7/4r0NxTrxTRtL4ly032Jm22tzfbrySbfedk82hKhCxeUaREUpL9Ev/vb2YAkgAJynKabrt769NGEgEMgZnBfAEYeP58zs7Orv2UuefXwWzruef8xl2tA56cT3k4W9hxwqa1RSd+6PEb1h7ORj1vbtsd3r+Yuxes024P+/2Ts7OzA3BPWq3WIdj/+Z/srNMbDq0ha9HniMGjechWrh82muzsKXvFk22QftVoWux5dPOVdxuyJPUmEx7HUTyZvMCPp0/ZhxOm/i1tN01DJ472ibOOoylv6OX4d4qdstjGYsuZxXbwf7RNLRY6C+56icXww/H8FXMTtu11LbaO5FOAyWNnubOqQKlRksa+xy2WzNwAPsI0WuZAVu6Nk/BN/jtJo7V10irDkb1gjyvd0N/ZLH7eXRbfz8/Z3914xfiOx7ds58a+G6as8d/fvWEtNnNnC54AQtfBNmHpgrOYuwFb8jjkgc3eLLgkzGgEBAHCjMafhTB3J2X6eHwWeVyQKeDX7uy2RKfPQaOzX0IfnTIPpu4xZGz+WSEcjtdObmF+xFHo/z/eoFJJj4uxoMfFxWehxzyKmcP8kLVt2085iAHzLLqXSr8KpX45tX41ilWodncPAYl8/UGbyNcfdD4b+VB6IAXfdajz8F/7/WeUhR0z3ZyV730S7QBe4ofXAXdm7tqd+eltrQB8UPPPT6uhpNWw89l0kjrd5DCMs+5Ymv1KdPsMtPvc9KvQsETHWlp6h4yfvbvjnY7Z+snLpPnTHbmzoefZ9sWoc9Ee8sPmT9HabP8U5cRs3THZP93xZ2M1gW/9GfKSGb1aiUbMRlUaNg0KTHy9y0kUcMA38u/1xh2xJyz1V9xrSF79+BF6KwUijbs//HcY9xG81pseYDYqPAn5ns39gLMVaNyMyaQFLv5su9cedGfugHjv3OO783AbBIc4TYBGlLetNiDc6rZ7gPCT1vn5F2Am7uBFveeAu5jzEKbnhO0Xbsq8iCeAc3e2hGfs5/OX529BeKURi0LOvn3x/ffMnaVbNwhu2XR7+2eCJkB+47vXYZSk/gzqBrc2+ymOvO0s9aOQeX6ydtPZgvmATZisbnjNPTa9BTMUnvAbPtum7jTgl8xPBTR+4ycp9CQiSzVwb0HCMdEQgaxjf4YQOIhXeJDSM9f3UNzaSqfAqmUr7ibbGGrPAhdkDVROOcCesKvlFXNDj13t4BPAwFw8A+4A4NH/cOo42CrQAw6WswCHSAhcHAFg6WrG/aABjc6H/Sbwe/eKff3mGfSDx2zYPwNzgwP7Bu7UYty/XqSiNI1SsLqjUAB0Wb999vp79qZvs9cZ1mHEK7b30wW72lyxa38HL8M3dzqDLnVQdIFFc9YZ5+8UAPF9AMpdcbant+IQ8Bc4A+lixYE88kEUe9BR7Cygdg3YTX2U1hLj6T4SAIFui4Sttgkw+DXwCpsCAlGp4SdiD4ktiUS9uuYRvAb8D5waoOh0ciAEcFGgV1M+c7cJx6a3bAaMIxtwcFncVbQNgfwwQh5e+zB2oOA1ODUTAQfmPF+7sZtyxs4ApZ57OwFACDwjjyV/p4gE6CFoniXwgBhrkoFZ7hxidgQD1mCAaCb/CL90B0NC98uPbyXYrNkmb3dmooxF+JjDBJUse9I6aeFYSZihXJxMvgsBQJheZkViDk8m0y0iYTJ5jv0Nvef081Kv48XAE1DnA/503J3rBzh7LPY1/L4rVRYOXjKZvKQvr7l45wy4O2V/f/bq+7/9NGHbBJQn8PDgMiv57s2LV6+Lgm77Egd/zn7e87BrD87a9uD5hHX6bLNFZ1MITuSHLnv5Vv4E8g37FvANCHii74qNL4Z29oqfnW+++754BRTlb3/5Vi+DWVb07AdTO+wbzva1H4ZAZ5jDq3UqPWGUh2wBknwKsq6QB1GYd+WHNz++VAbb7xOOQDeREvvqL08bhEx2+jVpsxVMF0HqCfuLqrnmOF7APEilyeTbQCisk9Z+AXwtlM1foEX4/TatKLxyK1BOH0QT1YQUBGMfCs0l+kGGT0tYSK1ay6iV6WngvTiFkUounEzCaN9oXlbfR2zwi17347JBb7N54K4TsAiatpuA8zVLHMAWoOEx6/AhOxcch/oWHjdPWndIAaTqj5nmiYCY7Ar191Ums0iZE3gUnKC/sLh9hZyHU5DmKFDaX7nXHIgNliloJPY6AO2RDQobSNJb+ZOkeHInOeETrBSC5s/ZF/o8BSgKQmOebuOQQavGo1wte7kyTaDCZuuDenLZ13/75hkD3Q+9f2SjVm40mzoZkLb4MiDt1zT7hRuREwM6Q5wMreeRnayclfs/UWwx/ZkfRnGTffGENUYWG9T1Fthk5aZfNHQjqhjDm74yjGwUQK2VMxoAu0cpfv1w9+HuUckOO66DRZtm8bUeK1IMAmJyOTiZBJErLVTCUFGbtDQ5YtCA5BRrgWR7LAWTMpPCKU4jEknAw72uWgQmgLN2PaiA8sX2/J1DNsMYmX4sa0oOz1hVaG3JsexVtD8jDLDX0TPQkuEMdF4I/yekrK6JVy0w3XwyrnKIyPz7RRRQkC+JwsKkA0PJzewpj8/8BO0zlNshEMSVaJJqyy7GsoEZ8ZbPvtqOn8JwGiAZFBQ9lghQCbFy142P/kfWaPj2PnbXIJevndU2aHQvmsUD1/MaPuDtYtRssi9B4w6ExT1WQc2iIAB7LJdPML4XN2CGsnlnyLr/92wE/bnpgpUMEzLw0zTgZ6A5fTeEX0suQ50pTF9AL6Ixkwa5MET/4fD4wmnTngdu6tCwnI/sXfum3bYYvvd9s9JDhHojAM573Qxizg8PwldvJDDTIcwAOLA5Ot2B3W4C3jrdEXypwVXRl+ktcgz0YpPYAQcxVrRoyeFXn9+IR9Dbvvq4MoqaQpgYpZY9eIATAT405GqV+qgPnHZ/7AxGQ8EMw76CVlS9YCPBYDQLCTQY32fOJg1X03f7DQ4f2tngukSzHA8SfvPPtrdOY7VBMtMaqEgyNiIBtQBTtAHvstjppuiAUpSA9X0qYGn9u5mrb3OAyg2J/pq3iCpzgHajAyoNtFFlutox3+hjpldoBNWbSIlqb7ZgQYCudDbjjADYsRtEww0OuK4Pva4idWFSv85selzDWHK+Tg7a8FYuM9GhYHtwcDha4zm8vQ/eNPCYzZ7heFyyEaYR+FUwWrEEMgMrETwjlJAgKD3heiXYDpxT/FTFBF8Det4VzKpjSnI2KQuJKOueukKZ/ILK73WTjpDxxNy8mHEFFXNMPcucN9AO6LaA88aTWexPhZfekEoJPZxoPk94aknFA9UAqy76yk0FVRLaE/ZRta40w4qBxwZc+PEeK1i1PTKGu+YrsAFWLjCcU1gYTogObMkaEfyoP0NZ0GIN7FllUpSrwowoqpJE69ZURXYvPQHe15/cln7Xh6KkQWEsI4IaS7ROSVl+pymCzG12EjSBE52fNbNY6SMQsG3o+UTwulJyZx0FrNysACh4/BMhAqUq7Y8Arc6h1MnjCqb4ZUsN6jd8EEBNdJNKSKVVNfByeLhdcXyu2/uKE5XY2H0AY4vZBIDe+e9zea5YsNKRAnUgqaoJ/dTZOBSGqXZavgleIrBO7wCTpdR+eU972VgikYB0qkCyoMhB3NX1SB21/tbC7s5Ffh0aDnSheG8hCgt4uiYi8zLkCfBNEfIT/ieGesTEpZhYzNcU5uSl8BNV1sxxCrqJl+7cYMuJieibDQ6AjKEp8jQ30kUYDSNzOTw3vGUrP1lRTHXF3TAp3F1UclmEFR8WgT+7cO8/lXmPY9zM7TqIcc06qg1UoLknWzioUHZ89sW7Nui1y6op+V6F6KXRgvTfqQai+v7sLZxcCuCbeMtLRXM/TsD0zxAO7hxwxeVnwGY2QL6uGZxA8HtlapgGh82N0gM7l8pIDkEsySEsn2WLk/Sqco18YQe6R+95l+bdwk0latfU+ugV50inNorJ3xIA2nUA/Dlzwf9ypn6aNCgQMVV+fjAvPWb0m7tBwi/NdXRCgiGROlHs+GHC47SBVATDBgxWdCHyGIK++tiq+XmncT5aYHMQB0psQY4LcfM6WvFGgzhEfyP2Xme1T4u8qM4+uvWEmglLhAKljzsy5j6kd2ACB+zD7G7CPriTP9+xHbDAFL48OjK+ksukfO0jk5ahsrpRhCiU0MTidg3GOE/8RDEdZ6mbSMMxsxU/ktgswie43PK4FFKBZ7JP6xj6GIQqdh69kytiZ2guvmcfPvzj0R6chu36H48mH0RI9c76xyNamMdHFIrEJ+i54AN8Gfw+af3jUT7Ft1S1MBkmdg+bCFWcl4qfsmyply2VMgSdaa+ignyQga5U2Og1EAgi0NlgMTahX0v91077ucl/Y+viFcmac09gaGL3qff1RcRh+Ju+3N1pMT18jdDlzfJT6dSUHxcav1Kk+DFKkWK4nat2QH2VUo1mEa0mI+vOuLKbxLNzN015iAr6fBV52tKuqVQu4w664+msPbPti3F72pvNjTsJjO21FV5jDVpVH3etTp+16JPW1dfbKa6tJ5zCQdkiEDvNo58gd8BJhCdv//osA/k1PKEQ94sf8mc/uSnKIXbCMBj/ahuSSbSO+dwPApb3hqBRIBFMoihO5YTH5UMwYTzmpzZBODlDMLiwSbIA/pkttuHyT4lcSBJGwtXSof2KV+dXO/mNlmjlYjCPoT49FuCmboIrfle42yXhGWwuY+W4CcanPtIiM5Vk4WAijyWWgcHauhLwYg5CBZddr96FDi3kJqY9Yu+vbLFEcXg4AvoGmVbujSk66OP6C3RxytM956EAh1EIRWJeVd98RarbzSIiYtGQ3oNBjxja8hB7JeBRzBfk/HYVSiUQgYbNIwbFerd9CPEClob9T0e87FgJ+wXi86HdTwH2H+8w2rFvzAJ/vb6dTNIoclZgIjtufA12V5gmzfcn2YSQjCt3VqmreidqkEGbKaJkM2Ff/00MrIjAqFSdKDtFJBpLTahoV18EOCg9xsnd7dBW6W5HbgA1joRGA3iwy7th6GGOw0rlKC72ZTVUkaqOTDbLt/qAlMSJ3Txh9XJSIlI+stfpjS7HquVSVo5G08Ecntgjb+CNhrNaWWmAUJGWhjqE0vFgaHU6gFT6MkS0vvr+9Svnmx9/eDERg7RRW6yYvR322dq5tUxPp+WnvS48BcaUW+z056S7BIYNBZappKDBCe48kjuf7JhfQ0VcxP5y/VX36aX6eAoNv4y/6rSfFuslmWk4ka6rewsSxygUhPwQAiXfDZKJqRyeFFcCmi+mK/SagwHFcK0fY6lyDRbtTACaLmz2k5vgTkCBoRxW7kIL0SHnvfR+yZ4IbqWBqA4w32YIX+Ed9BnN5xoqcLXky/lX/RKCgHRfxp7AEJP8MKTteN3xqCcmWZkbAk8QhggPrbsWewf0f39ZLqbOdagUBZS5XLQmsufxJ71GPrx3KhNk4FbRTtbrwcDBMPI9+6ZSBmb3l6CcTUUDKJIlYvxi1393PO5ZF6bxz3Yp+mT2dRBN3SDDAr499jqXh+r0qU5X1ol5NsIRPh/iP51L3c1CBvNdqZzBGfDZl7h5REyp1TawkUklFG8gwWSLROASZCiUbFF9B3IdsCVCPmc5L65czw4iOxGNgZc03irYbSTfZOyJaNeXo3U9z04EGqgXEmGDjB4lCISSkbH52CpwOZJEu2h3hRC76LTR/IOHz56bZRgJE7Epv65EiC1DYcijm7I8I+kntX5JbqlFnyzTekaZ1h0aZBqRUsC7BOmRJGL30eN8dzFFQcqSpV6imCRIp2sWIV1FhFx0RiRCLrItvSVaGEVAbsiY5USPKiEBqhVEFzp9qiLx/UnCJFnEisDAjpUnJOF74Qa4Spmj9Zx1TVKlThgNFWEk8NXvWWPE12AgRe6PP7145Xz3zf+5X+7IySRfgn4J8ADYTjdkkbLWE5ZKK7Hgg0s0R2EAElXv0vd2uY/ddj6A28tC3ORCoduxhHCHr5c6ckqGqalx18pBdNuXJVkEfhqaqnJ9rRAwpeaKGBIwzM2N0qVDWEM4BvnSV+RLZ3hQoHcGolY/I+R4JMXQxViKIUHKWmMqlw4PFkXF6QVDKylUWgeLjSIJTMVPNLO6JjML7KZtPOO6VMK+f1ahdL9M6rXbHZRJvXZHm2P1UqlzjFQSogvHc1BsKUg/Qi4VJHhvNGTq5Mqg3sgZ6kYOYAGNO8BGJqEr2HiYtIEua/JGo/AR4oYm5AFx0yF7gDQ/TslWTQVFJuQztzL3heQy2xZiPstZ3amD0LVkl/sZOoddFOC9NtiOnS7i8+Xbv99jfugOZVY+F5NQ8VxL01NzU0/Omict4zRWfVaaxlX/Z3Os83OJ3jgLcQcxyyPU2szcKB7IBl58qdURqJUFBnEyNoqTfumxnOo9nOsC6eCg0Iy+6AoeLuPc7Kgs6+wHMVV3dcXCugBM1M3gjTKFNwbj4oHyRXaqXTJqDguZ7Fzbe4mjTrtvdXqApE4XvoxNWDJP9JGc6IVq3jFR41AraZorrZBzsnZiXkuhQfEyYMJzrJFLjZK9mp9DesKuAZXf+Cv7Jttdo8KxZdAM/e9g795q/vM7XLqwKsDfs2kQ5VFvdLSgIxi+PHJWsIa+ygRv+vnl22YBL4nwhI3ggQQ3DymhgSICUBmzr3Y+Bya0oqWeOakI0T66VGFJO6hCsk9GDVWLO8YKJPlktW6N+OtRe6zal6K6RoD2enXO2zgvr0pykhNWSar0uzWeplKbvmS+b02XMiGU66/ljvCP1jw7146GsksolFYkLgxLuxJ5UTZ6rJpaTPe4O4RDemmv6tNLAGKO9kYXpDz6OFdJeTx78+aHV86Pf3vzhwY5ToN0hkYVMuoaVQhpFoF6UB2oQvpdeczZiPk/9AggCHhzgJgaZHrEiKrDyuQY1aGoCaEb4kLIG7QBaI1RoRCObiH1BQju20SK2stM+Cuh3ePiwvVSvmKhdxTzvU4olNt0ValeaoN2thiiQZoPSFBj+9FluVltGyG36lXEgGTa4BgVMfiXVhHVkZPzgyQcHSgeljFX9RwGiucw7o5I+I+HbSn8v/352cgk9x8m181V0mj97yHwL8wuw8U9LsO4R2sbvfFIBgHKyP5D1ANu2sJluGiPpagvY+kPl+EPl+EPl+G3cRnu22Rg2oqll8nNBf32sDvGlC4wbpCf/N7NBXXbsErlKETISuyzFn3SlgJ/tQ6K86Zq3pKpafuHwMxqUso4QsvEk9J5D/O2k/L5ZOUYjbY9t4H7hm/FQfYp7jtt3OIO1MsiMVBeyxO1UqqFcXnRH3V7bKWu+IiTUhtL6XXzUu8RNqBZLw6l6PmOqLd45uUxfnPybzNnF/meZag9fVBt70G10wO1W5XacXIM8PflbHBy83wp0w0P5vYcT806uEAudtQL3huPrAtgvfEFLsUcZD1cTZywaRQFyvul3qzjStQMvwIH3ghGmWFSONrJ7ofEMvCcnuHvS0O7hai+kPwWSjhralxJPmfRkLWUWDDUMgNLWj1RxvVQFr15EBvBCA/UJ6JekOE2aMtQbz1Nc6CLh7Fy+LAur39lzkcsr12fyFBs9muoS87NuomCCBt0cfdKa9Ad3z8LQHOVBa2ipspFpglQrFh98gSQDJyo/OslqYPGmoUvUNnVMA+yCbCUE0JIXIXxcb1UHZeJ6xNEdzGYh3L9wzjuPp4f9Gh9Y9Bv/2o8v3xQ9UXyIKZPPonpQ2J4MKOKHQQH+VyI+dbgIlt5q0cT+c4TTBxgZmeloupKl5m9bqPrg9h9I9hUcutOfETEs5jMULL9Tn6CHX+Q69dy2izWOfvL6TQDkDoplTmRbT7QUyIaUheWzSGipIuHwHVUmSwhKBN92UgrqBSo0PD5YFto8yAOXt4z5YZtSsk3BFP86CmXPGzSzx6mGd2HKZrNp2maZOHGHCwpmRyjyL2pFpQI17SjJZ4ZA9OfN+hgq9Z9yf2TyQtKW1ac1CI8izjMsD++D8+/gznrz3MN/CUb4Rm8Np6i1Rw/eDjKHpJ+hgfgSpZwohxcO4yeAZo5reGwdx96/pAnv295MuwTn2fhxv8N8uR+n62cUxuzpEp84ba5NmuNOu17tfnRQqAoTaP1v6YyL9my988z4wswA402g5JUTiDKTlyeR4guk5G8IadERdxvO8lGHZpko+746En2WWeNAXz676a066KOmKY3qMQbs6cy0tjrD3tTd27bo0FnOJ6OaiONebtKjDEvEavRQ7Fqn51XwmNgMqHht+vtX/FcXjYyKGrM8Khrk+03Eyz+3s2OsallywNlu6JMxtZFgk3KY/GzxV7Ssb23MgUjHgZyRdrUq3cbMgZarPt4uRNz1w/Fab0CFGA49m8scVKRcsDueczZdo05+biXAZtFYepfb6NtoqVtw4OkOahipY4y11IeDiWHb57ZSSbChac7kWEDByDSaxTAKA+KH6oJJmm5O9KyUOJZV3G+KV+HQOAFmCLrBgHLk4jYetZhSjhbk3u4AJZiZmY6bqskEtbO11+yNLrm0I24SBxMSKVWo26BNnaFySuucrwzwe14Jsyj9FkCzyhPoiTxKa2sOImaZeTiascA12fywD8d9XUFMuaAoCnmYQFmwk7BCLC7c9cPtnFGRJXb8Mj2hP24xpF/JfjuqYEpo3qGnW7y9t/wHSULMkCYLk21KNjWIS+o3x2gMhbzK2slp9nfo3iZgEjh2TxDHDSuFr7ncWLwq6Y8Ab1NOOWilglosxRpyAspHZtNZws8eesmiZ33ka3nzs2EZf2y9IJQLckp8CNNuJ9evfjLd3/9q/P82Zuv/wttQX3+vb+ivND5nnNlGa6ApJ4CVqY3Mqe6vkdZff1UJMamwWEGXUxvUoCKyW0o0li7yhEd9XigqxwOzFkCB0vMYMbDpq5gWVeggSJK98kP6w/VsGqFtkqCRLTgNmNH5gAkDWOsg2S6pziU5S1zOQz8MIDN4eLl4eIMOMWQBnQ0YKDcv7QD60UcSmwsoiSdsNP/go/vUQtZuBSnmJiYl4OsxW3h8AHln2eMLdMDJDiBQDY0NA4VOgPTqjftEy1ljFZNan6RMb31BN3hpIE52GY2peZ/zHqX+FLEqyXmCH0gJwsEV5s28rkBP8TsaDYzKBu0J9UJgrOmrhMbR3aimzent6NFUtdGvlBttKRGu/qhFtJFa3cNgomabteCnCOyBwcX4EN0LiRBt2vMEuRE61Q6JTvMvQy0aTRFwtHdR1mnITNnbudgsjebdhq7YQIagDfokDcT5/X/RtpZEw9mNS1vmdr5fI8VUpFEX2RLKHJ1Z2i+EjL9KstXgzsK1sBlPt4AkII0AZv62asXZvX+JwFP1rNAz/oe49A5fGnokaBy5Q0kJQ0ujycnlH4eX834DQxcAHz76tn3NkgO9Nik7tsmMj0/5rk5K/ogx5nfbKCO8xVFP5JM7ZZFMLYsdHEq8+NzgQbo4s9jp01Ja0U+sXM1mZgfooVC4lNLp0+w/FgOzQ0R1lTodmF12ZTQGggPDN8ocmbliRCyHEYTYalriVZlKZqVQjrkBiJak+VHO+2R5l5KPc7Kn7oNoDuhRaq32I/QzdrDnNvDFNrvtFSYkZtreczB2ziVqX0tln1rPn1Kub78SE1uRNm6ykl5Vx+ZSMZ0urL35exTOLi/k1U2mSCtXkcu+8Aw9anQFuwOI2krWxjA7MkTwFz+46lMBVVUbzZLrpKDtZB11MyFh5Ij5wmmAAno58IHQ7/HkEoKc93obwOWgKGqsgARjBnRmn+27q26PL7qrqZqaaRFoikti2WGvwKXl+XU3ZRJI8vnj2TOUrHlJLWhCklVG6wXDX3I8zD3HEqlC/wymaAVU2TJyN8A8li83tRa0v9eCA05GkweThJeTcOvsgYaj6dIToWSops2iC2YoM48jlYi8xy0U4MYRX8MVSX4asp0b6OlEpbv0lMgV1IcexuRoJadivpanj0vqUA0JlauQk1UqEoCZS2H+vMzEKvXmQdDYl6VtZRFjL0go1UR3tNoG3ouPAtApRUp1Mk/ZD+AY0Y3o7DG+GIIvR89xiubwJnv9vHuCfzVzPNkmNRCsc2Ntsf5eU4LkQnNFZ/xNgxJK6HtfJuZ2qjkz7drJT3aHhwJZyqG+UReOxDuJpOdGztR0nj07V/xkgDn+es3z7598ahp+4mToDRoVjMFF2BAQqlwSxkKGynKAIrGgRHcBniug9n4qHJDklmjjFVMwsz119N7IiukOnelGzMPlPggzdkr3Rhg6vyVJsfBzJkrTbQ6JEdBEf08fv66jBkKoyYTHEdJhmXp5AGuQWjfVaTxD3nOal3K0c0CunQwZkMn790oZnoq1UXMNhH5KUQ4sPiO2RgpKtrGzMYs+5esfEUcrciBRd1b4hGAm+eYzyVrIR71jhd5982NqhKx2p6SVpPxRNZl5bIPS1hQIXNl6m+YjuS4gjm6D229I8p8UmdGyVaefqyk8eXi1iQcAOWzxIT8Y0NvKcpYz0cZL2Vual2KS+TSCZvaG8Gw4MXQ22VmapRU4BJ1a67mI1RPFEwXyROp0xkAnWQGYHeGZxnPHzsCKfVrR/FLB1HLQkeMR83qqSpRnGf2epsspOlaHuOeYlMVY/CXkFoqU0CRnLgHsZJNwl+FZl5OLSk3DnYlm9rHoLv0WxJ2Ulho5mTngsKTzBg01hHz2KohqMAputsZ5pRCMUoszMZyqW19bGNR3sWq8VSk469ySw2nCH2ysQoNkmjYyTFTmI+VUd+HFYGRSUnSWYX+UTWLj/l0ie0x7asjLGhVpeA6H8YLSFGldgjGJUjLbYgXmOD64XHP1NybEqIlfZgsO7e8D4opQQh0Z+W9BNlawjrmCY93lPk2TSixGDwIUxL/9omMiKAvYsj+Ryuxox7tqxpjYqVeHhtECmIQTKTBLC/6UUbEGg2MYTRbVJDXeVzqV/kGZNUr1Uw6Btx0AC/9deFGncp7UOBbYKOThh9L8bGr3Joqe5AJL22tSP2j1bQwinELuhoW0t6WV0LvrXKRLAYGNIev0tPM8as2XR5sujzUdHewaZ27mXUYw65tU9lSlHVMZTtRZlJVcjkDi3uGW21xGeNAb6Oit1X5upmoUT1D8+nG3HB5b8MlNhQbEuiA8Ljbs7od4yxQuVOslBFHCC418g2xuFJV41EZD9yBpTbdBq7un/kJhu3entHKXCNwkzQPL+PyQYuJXRVNS4NHUb+sXuYYets1aDpMAbyPtoGXLV1QrI+WKkimZkt1mrX5I0XPZgs/5Gfoo1FC04CupsQ3TTm0WLnxki6vmbmYnhMcULl0li+0SGC0VAbSNYZ6ynqIH855HAvxxdBDB8Cpv4IH4Lau/DSlSwtnXFz3qcATq6aWPHoYiWDiIkrpRSVjN5P8udiSwkHGRwBf+OVj8JEBK9KCRO5D4kVX27CINZVByvvSBEB9VtiU2FwNrtFT8LgdjFhq78vM7nLlauimVEEoFCeKG221i9yQElykBRdr7WfwUpEUXMdNliZb/4WDpO93lYvr9OZW+c6PQgvUtEPQSvDr0jTPxBQzig5lilkiDb64o1iDgnmvydp7IraR9Ns01cdDZS3MNNPJYMrWwCalO4bUVQphc5dFkFja1Fvpa0LZso6pZXh001apKcnhw22NS0KGXmzuAyRWhAwtl/e1lG81NN0d37Rut4oILVW2q+SP5X6V+bw/n7oz256OR73Z2K3dr1I0rGxYKYqIt0a4h7+FH73+QebCQ6VRTIvEwpfnqyn3PJSEQRStSZ4Cn5PgTmwqdIjbdd4m6XNKi537pLwXTFklJHsVANF6rQj9yF8h/aycg8sX9+RqnQJgowFYar92GTRVVhe7VapbSsoL96oaJOGKK0UaNLEET7KFQh2Yjje1cfdY6LHd4Q0mcmOHBg+U7di4vyS/nxIX9DdicwnCHXVF/m6kmp9EAdDI0yDSCPM7d7t2p3cjdkbkUVA3lbdUahpXA/ITj8/o+OnmfMnQfEjydPDXWzf2JuwKnpFh4YgLYvdusEyUdUkNHIESW3uw736oZEOnnRKkP7M7jEEJ013eYiuEESAgUuY3F3m2szU8G+yaKBEXuwcJLSMWC33qxhldneNVZgIx8j7OgLuxvOQn3UdnAZA9AHOEboks6ff87gNyPnEylLSkUT2Xl76Edg4CsyHATk/h4YbwjU9D4LLs6bL0tGwsYP9ozZK2UxZOqiaA5bPyTmjTPPTn2pAN7hOl6Hsi5ySOQ7nEMN/PLXapgloVX4QsLz+APmYKQokpVCK6BPBo8aBA0iXFN6XLAchc5bNt6u8knwmhgLeku8hHyg4eWuXQoGXbdFjqLrm8q0DdneXx2N+ptqeMlRZ7dKwyy4vrCfbIp7R7TN7OgEZ0thZOZvKUw4TnueQrMSzo32yb/DL/tsu29N5L4UbOUBYzfb2fUioXWjoDln7qtKroF/SM8x0oio7A5xrlsTzTDRUoys4OBQY+1WBs17UQcDkGPrJ1w0JXJRoEJQjbFGkwLoYdqw8z8WI8tgade+1B9a8Sc1D/ltolocJwLfbraN3F60JBDOQhaHFJqBHqXTUOgH90PSROjIa5PIv8WPXFywNlpyQ/MZJRX6d9oEzjtvpqAj0Hy/HGyUPlEqGHYRwoDg/3EunyWxJg+YkE0Cf4b06B5b8uBXb/HhTY/TIKGNdBxI4hiSctqpFHWJyPqlqrvddOiT8Je9Eix0G9KRJN1klp0+5BYJUNvaTDQfHTJcqJXd+YwvI0oCbuTFIYrbZJzmwY2d+LTZZtTVfXqgJh74XmxR2dDgcKcYEdO/vhcB+PHUsxee6ttjyijqJP7q3bPqLOpn457MAcO6beJjmuXjbnjoJ5RLXwuNHUzsXfCYGXn4nAJen5e6fw8n8PhXf/Oym8+ydR+M5cdFf1DnCZn52YNJ9w0ZJ14OM20jSiYBJ4/76btDbLM/zViqM1+J2YfOyM9CFDhYmnluimxbZMLt5p9++Na+rrURQio21+eN1xwrwo/FMK/27BRz6jlRW72nS9cBP+RSN1XOj36zfOsx+sOjdMvV6WknpJvpxuoEW9MWZr6aQK7ww071T6i7qHTA7axim5aK1jX3CEpXC4yhFqQ3b8cKUHaMrqkO+Frd028iCP+e6B5F1+EnmXBXlLIQ8abHZU5Tci8PJzEPghctQw6Pug/xNJvPskEu9+1yTe/f5JvPv8JDYuZkyYy96FFjMmXsUYq1wsqQUZPharFco5rCw0nEHL1Jc4Rt7ptHtHrPhqKpyugaunaJSIVA8nv8QI2Ojp/Y/G9tJGrS2Z8x7GFHgYjAQeOoPfIx6Wn4CHu+blAfsB3W6wH16+RftBoGB4IVCg5ZA4CgV5AvZ8JLV1q3c2/BNRIlgDIw1oyD2EPUYSN72LfwXc7D47u7gzYW5+XbDLRUegZNB9IEpmxfXKNS7Wp9p6m/uxR9sBljsbN205y0ZgsXbz2Oq7e6tnqz7G3RcpT9LkfI25IG61/Relguzi4377go+6tj3odEYXNRcfl5tqezDKhWKHDwk6/Oj0e/JgMDnKq5Xr4Eozd0i9ODMgvZwdyBr5qV89v8Z+EQV0s3eqHnWaLVwf92gkKR28mOH6934hMlL8fP7yXMYkKdGAemc5qKi4Go7EgJ26Hq8eqH0ORojFXkU/vRBntV++FVFLCn0W19HjdgrKwgA+Hi1t8tJNxufKNcbKYiotfYrzs/ri5+EFTwFPXfW02QtclA3xqji54grNsvfjxgBcgN3KVX1gt4CJe5Cx+wJcFGMW9XRPadXpUgEPRweuOdu5wRYjpGI7oMAUIoB+izmB2Cjubf95z8OuPThr24PnymYOGhnlQeErPP6G+dLPitcU+BT7yoAQ//EOuew9HSGWL3IKRhCbyxKH4t+4tc9Z53vicNdeuoij7fXCyQE3muq5XnEeVkaLm2gBX6+3UEWs22YHV9V96lqyU5n4V2Q9FdnsV+6NM0tvaBV02KcTTRbrZF+62WdbfpFV9F3uAnxmORYphpVsnVY5fafhwF2+4UHd6aBU03LrKW8RF5409T0RWQMpAGmjPo1USyFadAHjOVCpY+MppSKjLsCcg+1vJ5tY7OssmmzwSkbMhwx8sWsgNjMfGBCIN6UCLHWUS1P9LAFQB+89LTXYHW7QrTTYkAeSt8j70gMhbQ/0vmg1C6D9StVZVNSrohBTDmO7QaUziR8e0W6Yt8vPgX4XpjwOuCslEh3lF9NZRKToYAOSVCzrvNuwj2wJ/+/e475flEi2fkyDdiw9YTs+++JdG6h5ySQuJd+9V07wUYb+tm1TjdIhPnEl9mkG8l1agLDtRgps22lWoeIftHxn20SP9zbmblDPF58SIxE0qqHCEk2aJVCymr7lxgB4mQMWVVTIspERdA7Ttg1Qd8dDvVPnS36WsaGjv+hAKyOMacupWg0fqxNOLevLi0YzZlMvvRU1qtyollYAVt7/mPXV8py7JQ60QnHcDk9D15xOBTUPONFSSEwmId83TmU+AMRacRZIaY2OFMwLcI4pZ0Ov+5QOqNr2k7xHMnLRNGZB8MT9q8Vx4EZlJPnJ4Pz96t6wrBOSsMWwQWm7c9RHdP6a2GYyIRaK3T0ovDhNGlnvcd0XQAJkMK/SyQSTT+AxqLxcnFCmztxVz72vMQ1+3gkjnjwhwcQZgAytyrZwKNaqC8FVW13k3FeSAEwP1RYiWWuwPNhgmTcobqfZigMLhfkjLR8UWCJ1Gu0pE6m1yALA6gbbRGU8MK2gHx/xvAweZl4u5Zed/MTDdnKHV/atJuXJx7KgXM5UvqJt2dUZaeapXPf9chARHQYowVCERn3rcjRPUmljIbWr0X4VZv54k5hPX2iHQ2petAS7DNjEEJVUbYCmsgfvuJcpISA932QxPrpWAqdAbi7mb1FtInpARygsJuYgdc3oh2tLampqlk/pI6FG6yQwxS/tIeDvIV0shUuM3ZzJrtKbK/2q9lpaqXmFf2JfdzM6UPqbd7WSATjg1+7s1tjpEoeVXzwrPdjNqoeoS09K3K4XlrGilxLmqg3yFLwGWDoODfkxyr2rRfLnmXekUXHVFdzPEPy7RtNUK1MdeLTEaFBv5A10OlwvjRYkfU+zxiLdsekVWGpK9kG5dg5oWTBJdSV7WMeWq+8OVd+Vq68zdxV0qMKensqHnrpC5+205NJGTqpbqck7Ih7pPqgnfZt6awXKdVdbhD4qfceKVuk3GrcbmXSkb8gQkVdqlHb/19TP/PzKGGsLDIOnpS9Q6vOYczHgCiO5ScLjFPRqmjh8o4zyVA5ffYMgpvLkEcbzlNv4zmiUIoC3y+6FFb8fKf3Kz99T5r6Fu84OEXDKmox5dbdgiNxSTMsi202LOeE9YGx8MbQEFB+94BX3fDzU1R+jDU4FItbc79BdAJ1B/8LqZOkpKwKUgkxcyCEK4eEOSx7zcMYprMTkPGMK0vjmiwYIoMDWU0SjSY5HhXMiZcFxMR/wtEqxACuirZNJboFOJvKkcc6hqBvR3sBPVD6UCok8kpY4a6KxEEh96JIiww684X51cVpePPfKCsQQwKpZEfUqsMry3qvqmtpxlvsOg9ZkOKux8coERIJYJiS9+OFZ9uMnqDOZvAI+yQAcVAKC7UZiiWPYGedJUfc4WzqUrt/xE5x0MlyeRsBuKcwS7hXhTPiG0dDVGspdPyjYUNF8Zt2WP9XYUEHPQaKpCCwjzmB64GgacsxjsRA67A3+jcZ8kNb/H+/FVZoewQAA"""
WAVE13_PATCH_SHA256 = "a94945d37f379119ddade5ee46380df8bdc31601995cae7bc02f31c253a2e415"
WAVE13_PATCH_GZIP_B64 = """H4sIAAAAAAACCr0923LbRpbv+oqOp8YhLRAiKUqiqJFnZEfjycZOHEtJasrlAUGgSWIFAhQAkuIkqdqP2C/cL9lz6QYaF1Jykl1VYpFA46D79LlfWn4wnYpOZxZkwj2ahd7Kd4/kg7tYhjI9msjIm9tJKiY7bx0EkS8fxHmv13e757bdPfXOjv2p6HW7p4PBQafT2QP34PDwcB/sv/1NdHrHp0Or1xOH/GEg4OI0Egs3iFpt0XkpPsh0FWZ/abUt8Sp++Iu/jUSa+aORTJI4GY2u8dfLl+LnA1H9mUvXd9IsCXxpidRzQ/gVZfGdcFOxOu5b8JIHJ5X3+fc0i5dWGU67+PrrxUFHfz46Ej+5yULItUy2Yu0mgRtlovUfX9+KQ+G53lymMN9luEpFNpcikW4o7mQSydA+OPwDgIjbuSwBkg/LMPBglxOZAeqkL2SUJVtYcJzR036QLt0MYCYj4WZZ5CTxJnWWSTyRIkhLsFzhxcutiKfi9h9f36hXAnZiuL5YwjSjmUB6msGLUvgt3nx/dSZCdwXbWgK0iVehL+AFxXPZJoapTKcygfkp0KldIHkaJ7QNIojExx7tCvzX/VTdX5p4C0e2/3ph7FGxRXc2LdOXXuxLWm2rmFz9phPKmettW+XXPEfKtcS9Je48S6zh/3iVAVIdpK3UYhLzg0VOQstYXQXcysS5W1fo6XNpsnga18kMc35snSG/nJ/C79/PLrBRr5M4TTtAHN4dEIvLFMO0QaTgxWnGRJi6C4lzq1F1Di6U8Li4FF8DbQA9j0ZRvGm1L8pb7OD+dm07yCSIn5+LXXt05z5n9/5PdvD37aLeSb9BLqeJdwQLA8YI4uhoEfsl0dx09yCSGzENQingu9RCmUV2l39s+6Q/nHhdj2T1kS/XR9EqDMuiuRE2ElvX6gKlWce9UyC0g8Ojoy/ElR6HiwuALOJNJCaudycjX4CImds0jgf/5K5hWn2xkG66SkAqbeZboJ0gFb5MZbKWKYiPSbyKfDfZ2uJ9ImE1ocjnIuZx6KcMy4VxmejZZ39G0YQEiIsOvwQQSZDNFzILPOHCJHjgcZfGLRXIjQv/ZMFCjmDSDDBxYQ0A8bgr3ry7en2UwuwQlwj673//Vry5fvcOpFcKu4zyTg0/6Z32bXEFXOXOJHELQ5u6CbxvqjgD5KKapIxmII9BxsJKgaCA+hdu5IFAjpFpWUAD5BjuuVmcmNi7iRnaKookiEV3izOEKcBkNsBaKcnVVQZoxWUTWgElK1gCsAbMOgX0bZSuQHjZPJEw5e0S7qSgLzJmdpjtMo7SYBLApe3ImMAL8XH849t8x18DDsefcCmphEUAvlOAHsPOa5Yh9glwbKoYwxZfKwQJXgGsd46agHcpA4TD0I2bLOEXCv+5i3SykIsYKKKYxvW3+TTeA5HxNEgiZYA3OQNV583jVEakOcapDKWXwSjEDEgg0HtSTwM0DvAr3gBluUpQWaJug8WhMLqD0Yxm18tWcHGLO283IwTkIs9ko6UmghFzF7YjFguQxMESAOC7FvEakKEYQk/FIw3JJJBuYJOfAQl6c+CjhAgoZ4RnQIieu0qlWjPSXiKXcQK7r4H9WyaxAEJO6X30bbLNZHrBpOGFgJ6UtxzYSpqUljMePMhiFZgKySgMJkiXEhaBs1xFwTQAfCE6bfHOjbbifiWTAPmYzQHFq2y/ELwYqB8HbQ2Lge/SXuOCYFk5RVlisiJUbnEKDK6wGOCRJa58BpzM61ysACLODZQVMJcnCfUM2lXqKScStY3fRbAeLRnCAMUQUI3JQFnsu9sLjYs0w4HAh4XEU5Lu4BD3BMRonMjR6E1I+vVCX/fwraORn8ArQPm+Bll7UboznQZw+QeQyoEnl1lSvqvMo9HoG/pwIzOCfAQa+7tI5gsopCUSn8VLKaQjqNhFSgs/4mev0d5kJsR9JH4mtCI6Dc7+imYlXN8HCYGkgyhduvDRZ0BuGEczJR7JPiQMonxBCgVeh012l7gdk1UAdA4Tc2E6qZI7cxelgXjz/geY3Z8++hLR1PpKTlYzS7wG2MClr8H8sMR7YIfADa/v258ODperCe7nystERTaBIXHIFs2R+J4oDm2DfGu9+Sq6U/Y3AomcjGTXiHV0/ug3P/JzbggWjr9lYkU7FshLmqDEBxAV42zMO+DD0uMCyhgkoTNxYTcPwR46FL2x0ggLSzCTKwG2QWYUyNxgTnkxmO0geIxpajjVafIKSeyWFkVXamuS26O1G66kekBc34Nsw60ZqyfGMBFpUA5OD9kKjPsS+Lt18xuuQ7mQqJRAkdFLjKe0kVV95qsAbUSgsYnMNhLfDu4BQddEDHtB6CeqluoVVchsiVWB38TTDOwwbZ8x4aIOjFezOch0AJ8gIUYgTnJFipMvOWpEbQhhJKYE/ldkwQA82cepj1eCogh5DeUgLIdQI3qPYpvYVcN7BUoTBDg40MyFhVkADF64fA9L5FSYz4hGKPkB4nONTDrZFgCRiollXZBxMxmRkEfqmwYzFqks80KW4DBfE2FdAzngfZjWcus5qN4puSKwHRon+NPCG7aiNnEk1FdNTzbsVavXbqvf/Bwhu8qXuLTQhSneax5nztJ8CZDJptBM5oYpG1LyATirgJei0O5MVqhbgMqWLvjPW+WNv7690iTIji0jBc3R8toJiKOf3r16Wq0hEtTqWQI1LPZmtcBVjVn4gH8TgfxY0wbodcLW5xhIm6jmFocqKaPkS4ZBhlmo7B+wAMl22Ro6BC5814r+1W+zNVxAY8SQUewmaLKrx2F0e0S7UBaFTRLwmx8LeDhrpTVQ32YuMcE4emE8FrWiw177qD8uI50Xg6RD3qGB9NOBiXT0RSPwReEy6NokXrRKaAevNB8agW1XGabn0caJwO1WhItoA+n28w37VSvkn4jaDDRirAullUtftQmpVPAHbXiaXAUCarWQvpWbeoBfD27MJG8hKAawFzwyn9BWJZJkeDAsmUnmeorGoHkiwhg5jeTOEQgeELZgU5OIS2EzI0U9ynAnkRJvGF7l8VykizcrFDAgzTZJjGEg9GrOHgRFFzTDoNEJoiRcLcCtmTJAhh5PQxBUT9b2lig0voyAISpOgClxb8ntUoKQrOyBkn8jWgIyNC6jheuwCsZpK8r6gNRYBse2ei8HQ5Z3FXFsymhzU8l3kZBRFBXw1BQ0vQcJbQl6P4hDxLREOeuRDUtvmgGPLdXk3ty7ZyXVswcRN3NwCkSEcRoCRUozS9gJCeNZWmYlHGgw0PMvU6RYDy0sk5MWGDskoWVexZ/KVEYjxKS4fCmeIW8+sx4ZjUuj0TP4YI7+tUEo/gOEzAKdDgxxLEHOoEtQ0HZAjAL4JPKFFUh3kRIWEP3INGrtsG5UO/ohXr9FI0bieUWvNwmW34COQrjQRJQebD8VP7XHtd5sN+Isl0tgtKAjG6xJDrHLBLIGLxe2FRAlS3dlYSFNTuNVwv4jIA0MA9B+12+v3zmv/nl7fTMihFyKwUUh/4ANEM1lb0T7wIH21kCHlZyRd2j9AO3j0tgNSUtCETzkCA2zydYwaSxW5eDEhcEdWUMMjJQTOcdgTcmHDLVQkNniFS6DDKcXwAEBTOwFS7eReNuHzdwKCkYBHoBrDJG1YGdU+6tzVnj6qwTDwctUBIfgGVOE9wNGGVIAgl36j4m9r+TUBWTtkH8NHk8MW2Iw/jsVbei4nrcCtLsZu7Xj78XfxDf/ujV16P2ds3C9lLbRegKI9wDiRxPAct0IgO1eohoMD5u2El107qtPgGNijLfYxmFi1AqrxtI1qHdVqD+Sq2PC5cC1R+RE4adUfBzfAAOPRgrG+FMN7roK97tVBn6CArxJAtwLdJQrYRIdvtFvwCk49FDDS+JVpl/T7FtU95n8BS3RvkwpOm+DtYVhMZRMGFO30AgunqnFz0YjUwCWZjVFrd1qloUW7cOoKqZIRO6YsrbD8kD7ZVWW6TumOYZPKPMOHqBhFaOvOtqPMxyqH3qxS+DCHf3CCgSmLoBBsZ2SgvBID7wwoOfLeWGIxQpAU+5eluV8bXLKIn3SvHe9c88O4E/O9Iiqit7J+bnhXs65xnoahwAbMg4b76733iUuaH5BkyVwS/7CokFelSkZF7XbPyAzX6FF+2QKEzvfaQg38RNIAKVY5IMXrnwOJXMgkbKtFIJnsxx0k+F7spLiEK6yCJV9X15AIToeWYbaJb0MtSPlr+u6ffA+oJyfrDosbIaiq4lESPYweQKh63HWg3IMGGvgAKrNago9Ygq9t3TwUjzPo5d7LawdJm0wzbPEYCE6ZBU7MnInYDS02gUOnj8XJrvYQeoo6pAOiLOzhqGlsEVbXF6Ks+ZBLLYuxemgdN+Y11keAXDS1ZLD8sy+lfhAW83D2L1Gk09tFOUq9g5G87K+qx9WEW9XY4hY5T6I9HS+ATDuB+jSm8bZ+H7M8YPGkIMlxncOmULjo/FafcrDU6G7pegUXWZw6EzjU+hYp1LDxjQDBZp16khkuLulCBalNDlmMQZJMWZ4ifQkBe/HH7UAzTO7hrj8NCbTCxYeb1peGCyX29Eoi2MHXQkHHOcVRRW1pQUkrNCm0s6YIgWixei9kkpNtM13QEoa4Xw9nHHTdGu9+xZJxPplhbymWzt09sGhWRlQoSBLqKTFS01mqLmIHi4rrMzsq/UuDlOhsUvlDy0Nvt3rzORYG41qmfwZ+7qP+lP7YHA1gILyq54wTdYsAlHLKq54xRbTZhqf1R4aV9a1K1hZYAAzxFH1siZN47ra16aRhZBqhKMKESq3OOxdm45ik9rgspRS97FSgT58d9fCzW0rEfOnj9501sJMDjLNIvYpq5PqvcfYFYhBTHm9uDA16PcbGfXtk07XPnmlIoUUGee4JkiAIMLwTX8w6NA8sQZlsQS9gbKMBErhwN8DLAcn36pZn0YwvmoWVW5R5C9PA8F7q2ZRnnvpWtWnVBakN6jdKVIk/cq9IhFyOmi6pTMZtbsqBdG1e/2TPabRnz7iRnzKkVS2mR1QioBmRweBHfBV6EJ6vwKxDYis2OykLC5NVBvWJuZTksyR91+0Gu1zC/EJYrg/OMGQqfkokMKVQGstURF7z01U9KbIh6GLDb4nWMIROVNBZpdnl7kBzu7J27tnd/vd6v7aNi7KwHTzynEODSvnhXe7YH7Rx0YUvMalI/3nKWyO9jIW4uROqXBkAS7Bsusu1ecgoL7K3fT9xPXXabiGDYz+N2Kp/GwjDRVDNO7KFjkgahrMVonMK42KigpwgK09VRNFuUSROjGKm6jwSLgiRes22V1xRMEzyuUFapOEW0qzJb5ZhpRRUQSVegRpQ8BcF2wGFHi3m7ma+DE36chdcUgDEzMDDfmOTmg6G8BLr+8wlho4HP3Ey6rnOBqByfzcYHurWZfvEgcYkCDPC4UAcsK5MzzvwoceMsPpoCoLADH9AQt4lYlgIcC0bz/2EhIylug5/eGJc9I9d3r9biO9vEHvIdB5C51PyR0ZleHXUb50KaWf3x1x7L8Adt+Q4y0yAOm8IESOKwSZyh5gFVq6Y2vJl1BhLgcowKFHsjn4MLS5wBYOauIUeYTdws+T2jhARVUaN51LSBqtriqYGVlznwVmD9FQElMTDYLOvzzj2F1R5CcDig9v3O2zfcCUC6ygaYf4hdg1g7I8MkDcl0DcVwTXs7wCKqGaWJXHScAGTfxQpumzuhRThH/LFQ6KQPx4EUQUciUycaeoH2eKZMHQOLGPxbtXeRVV1x6ewfcSQJ3rIFbSQktZVTuY6YtWCT0vS0tFlj2rsVKVaIsYhaPCIGxnxBQtdTBO+v8leio6BWWEEUGp6Ry8XwROjK935a/r5i2siqM8lGWVQD+KQNPId9JVskav1nEdUjcORlvQdENrDTZnJ8M/agcoo3S4x2Ad1swAYysetQaaPBbRe9JWfbYwsn7780Q/VS5GtiHJjVliLcUXEpQ1J983Mfm3KaWwRICpR0wkcTzwWaOh8uvOMm/lejYWeZfvqWru4Vl36Pp92x50T/tD329swGl4ulbmXbmPRd4nYBif9cQh/j4+wY4CSkHkDrbZLKD6YX49EFyzT8VJFOn0dYkJO+KdIuRE7t0Y6+PHgt06XZbHoor6RnJorXd9kHPKXmqPxCSMvTvRmoNd0OYSTNqZOUo3LDrJxAbUQfG8UTGDoXz2qj9mn6gAhVOdMIAeNWpVdD4xyNICFNV54kQeRIsrAkkkw2bKhwCElzLf2FWhRCFoAMr7kUhs2wWoD/J+FSTKw9GhtLygKcbiVZVOAoM2Adgt4EkSnJxwiKMCVt6HwXqjbYtx2X/PA2shVoWkmd4ZQMksm2NdQsesA9NhnNaYJpbX3uCGtS/Ibgn+DVP3t5G7wNIAs0Qbax5LM6OuEOW0KOShuiK9OMVacwniDUylQff8lKunOEWf110prD0pYNcxIuX1ppG8q4Tz+8X3UkQvv1qNqRWtRMpHar673nu3HsXr1AQy1S126jGC8vXmwF/pMS12G0GWqiQ7lcDCtHQRt74+0CQx826178gIKOZPB1POQ+Qx/HLLz/Pnedz2z+IMY+3d2gBzgRysbxphxOqrt4u2N7ayiYyVZVBeXRvTB1gMADovh1LpUuLeAV5VYxCzVR5fxBdrl+8brjHNNdxY77pB0cfa1TwA2WlsYqIYZO1WHotsfKhojOrsa4yq31Uhyfocgdwah5t7UhnQNroxja6/5u3Q3WGdRzajuuLGTWjcgDrymxG/A+nNCN+D7N2IbkJyA4J3IrddeLg/zSU5WqoQDjQWsa70O2++vzJyShs3VakCbEXKdA8FVsQ0l6WaVUJqcKKK6QJSh948jlMsqKNUPBeyUl+B2ZFSQIxiga0IFE6MqcsFLQvQ5Vh9gSXOefMrdzxQEZR5NeP+IA0vKNK21LCaBNOsnJat5yKL7OwkjsNaerYQfPWwxGs3Kio3VfkQxp/qel3pSzA8Skr4rw1YvqGWDpWN5IAG6/nOTMZcETh1Pe7bQBVPDRtoShlFzwU0/cwFR7Cqu+emd2mlpL5DpU7U84vWERYM11DYmDZVxXgN+qYRu79Nqtet2OtaB7WqIi0Wu6R8mXwAbsWKNTBa4GX+Bq2+q6NXRhjxD7GID3+nRXz4h1nEaAgX4H6XRZwbwmaM9DdaxLkhbJZV/EaL2DSEDcz/Ros4N4RNrP0ei3h/bX+VYllovlYNjQntLjV7UuuLSRjcZ5YT5mikqGys2ydg57XPEgdmow6+HsSoWWLGtW6NthAWmIknW/Vij1Vf7/RWxv0uVxv0Rc3F5mvKtQYpeHbal7Z9fjaV55PhTtdaPVVzqdV1as4/t07FIfyLjfkHQreMcqvO+JP4n//6b52KwEAmhek6s8RdzrFB+Y4TnapKij9vI++IREQBLaVzNDQ47i/uwEZh8RHfEq2rrz50ut2e8Fco1bikB4htFgIdel+mIAC52AFzt/n2XxwIfY27YowLxNLGdx/42I2y3Z3svOYa6vPLCvvecXcoz3u2PZ0OphPX24n94sHaBhS3aA+GtAdDPhyBctF4AAL1fo/UqQSwkAPR2LkpSh2YJmp2NnSKvQ2d4pGGTppyd0Bz7p5Ygz7OGrDcomfayAMbGczmmUMmUmsDjuub5eonumaJaKTOLgndCTFIoSm5SkxrOlR2VDXDxdB8wgMnpWZYLMsGWCWAzHrKVFJmeRAacqpifBOkOsFGIgMsMp41loYkwUPe6KOumpW4E3iNamPhnAnpi1SiBTfKG/Z0go80tErhndnnpSZ/3fesyo1woWYfmJjAgE3gZ/MO2SNqwkG0DtKASo5ic5DZsGKsGWv7AiQWRiLVyY9vbp3rt7djsZy7qUyLAiiwS+MlKWndlxsAWhOwjSzVQwvTwEZY1nh3QDNk3WVymRtjC9Sa3IoTbi2utaHm3VJlYUNVIee3uHGFuBWUVsJ1y7jW2FLnBqCpPclL5KM4y3eDSu+p2ZvEhUGTxr47RoSd9UFk1k1zKCP/eu9ULswD35elJ6bAMv6ILD1dvGRUPGLEe3rcxwzWLzgLLkL/hWaEZV+DC62URM8W36t1V+xWTk3kzTZxB+Tif4IDA/fs4jWLvNb7kl7Zwjawe7NWOZjybE2LFF7cxy1LAx97bF2fz2f48O7m2zhZ0Ge9G5a2+bA7ldntQW1K+XwfPvJGzdA4FChCiMxTZnUAT/pQzRpTsDhzDI0cikqlNQA/tsVN8PaH1ow294VYLSuTxA1GiptgG+rcDdfkYXny0Tfy3uYh8Vo5I6PqVgc0qeiBTpfBgyqoYxd2FxBo8UJbtG613LaVz/ApK8c6hHwPC0xUrtYxkwbhCktJmW204AS3B32nDBP/LsgqL6bTT9I0n1NaLljZix9zEhWU5bQ8qJCUSu7hQSWKmPEQFqAnZX6y+AkiuKOmoaZQxwxXdXEBRF5ITYmQDktILV8xqTXisy5mIR3jJTAIlhZvnay8O5kdqAZPJT7FeJGSBT7+Fmh9TFznKuGrjwNB9ewLVPqTVGL+lItiqImGoaG9DLSB8t0mrXnWH+BBSGenPes0z1qAauQGiZ/LRxIxWrGXwOaP3ABQGqNDkDhIfa6N0KkyNUh/NaJRujKniFWKVtH1vLPhWZ0VVH2hjB/oVUm8lE6abUFLXV6KD/DtBr+MRt/CiMpDySJ15JKnqD5XRhjhIy7znNp3a7OasFXMB1E9HPbRQDkHq6o33ItrRa1+Ah4J7qsXLzE3vsxLvZR0E3jIl0lXbbsMhMpQdSdw7rYC+bboCsh5nuLpgIy2yvuNQh7XOLmKz7nAowTItzLS5LrCGQ2VGrT8YBcYhUdZZJzi10sxND2q6CKwVJEBeV8I+jQqaWv4Xo/kb8u1XFF+ttRhU/SWUxKEqZ3j8vyDps49I4sMscECO8cXKYzq8Vo7xtZKWCvRzNJ1TD6XaQUFSqiOFCNi5lIiG9x7ypbXDvDi4mXctUv2H9UTH8NPSE1skx+fW308/qo7PLN6J49SfaWug04owJgKEmARgbnEPTmMymeAJFR4tBNcfnZBEXBoPLqgFDTYCa0c7LDrcfcnncFmhN8qsYNW85sb6qtrL7V2P7qcYuHNzttahJEpetcKLdFtP3X4+tHh8HJEh9WMK4OXcjLfObDGELtfqtj4EWB7sjFPy8o8lp0pTkh4dHEtHShr7+b3PGyTS8EdOCgd7Jjzfjvnz975CTnN/eOh1e8/iT1zA7TxhaeDhicu6ivIhUeqdWdJ3GildNjAe8bheUp3mP4z+eFYdJB7z/kBGY3QSONMZAhCxnMjVfOqPEbtLpPZRP1a2rBy00ZgVA6LsyIjX3mSG2wG993kTkd2tU03ovP1cPjwz43gzKP30I+NYu1i0oFs6pwyVcqm+t/ABLSb0c20QgVjj4qhxkqggtjMobq/xTKosd1u2DpCBxXafgRP/+r1J7Tk80kVZahMCDsBEAYaIZhFap8B5/rtLQHa6403M3KtxUD/aOG0c8C98+gQZW3vHXNno+fs3A8dpPuip69huImSRl2Ojjo6RZfiF/RQVim5rQsOlb1zs1928byJzzQjXKpom+q5XNgbfHspBIcXwSONjKhb+2IffCIcBT7SaAGoCxtcNwObdCmIjCsqPjgY8pmvJ4M+hzVrou5AlJpyhNmU06k05fw84Wnh8SNOPJ2mEgUNnvGwWmC94tChxGDSPzl1lu421ULw80E002UZtNCgVZCUrZrR6Cu5vgkDT9ZHUFnkaJRHQXOVcNLl2O/J6ZlSCSYahE5V0Bmc2PqIgS9Seyk1sfIBdalcunQaFmYqVOicg6MoGSlmVoCig3fUgchk+6MNNr7H09QoljkNqR8/z6ahDL2Jr9QUOOFdQFNnhaLgRH3SdBzN8RW6zHISx3diRsXCnAhLtYmomglWUaE8uG2hgKZ1z4zO/GLNQTlu4dLJBSlsagSztcXXU0DIFO7MixBn0RdCsJZL6XLVPtqymFubukGYqqSdflU+ZzpJxOUjRuiGfMhK/RIYC4jFaunD5HYUyZtUxSW2biKp4piKVUmcqOAnXcV1OvJhCbK+Ui5f6UmzHu9CYye29f7D9d+/fvvWeXV1+/ofJXiwrpNeP0e9yvupmDYeupq2K208aL9pZ2SHIMfONDE8P1X/DIYgVUWWrOSO2tocpAUU4wzPus7Jef+xsaqN4vi47wzPB06/O2woV19FHO/Eg8v9WBq+NQXvAiwJ0MdEprq4gGI5uJwSOLRIiSnwCYwC4gkyGBpXHRqEvjiI8qN04c1FfNasYP8crE3dEI/0elmg3SzPFia9CUVvpkZwmLUc8rNUp4ZuzmCmLnuchsw0wTTnzUhSHeFR6dm2lDmr3NC5s6E7OHX7tn026J7Ls35j7qz6aCl7Vr3JB4wP1AHj8HuoDhinTXdMn1Cdb0DeBzkYJLTQwiwQoKiMTp5tPZ/FwEPP8eRiS1y/v3HeXd2+++GtJZ7VgGNbB1dn0IEnWP5wQ9HC3oSzGRMusOhgxJyZNDWC+63xLHSq/uu4rQ4c+mOgWboFnQ8H1VmqcUMuvc0BTGICycduzulMNSVecsSR8rj+8frDPzFwYLHhzBUdRbNsFjO4Sn2GrmjQ1RmNNR8Ue+DCj7bORTM0VXVEjgLIpNQoRMF0Ep36oNbzJRZ7JQG+K18CeiK2Pp7+FHUvnk8PSlj/PYdaGt8kH4LRRD/EQFHqTsFn4yxuihYBH4/iJC72QiUg03F9a+Br2wWAWdJq256Llv9fVsOXWJGa32cPjWKH2vJAPrDnWey3fDr6ndxuZNG2vYo2ibukw+vZhmqKinTKJ80X332znM836+9801n3y6WDOwMJtcBBAQGm3BQX3VEcvAO0igaX/HaS+UzYRtoDmYL+ugRLZ+PkX6qsQS5KS39zIgfmokeJedrUON3BFn3VRWd2XqjzUPOyOIrcFsHZvNhcVb4gl9DxA8RlWDtEJ5u70YUSQspGw78YoRo6ZJobHur0MWX2iFnC6S639qcquLgNWY4O2PNCN8iPozI6cVi6lgpr9gR3jaCu8TcEyicD7IvnPh7HrcZv6yOeErEtVZE30czOUFJpXflVFCbSL19X9dYG65XOcKj2GtXOLSgDNc+q4CMtitU07JEOYKq/GfH8zkImRt5FlmVO3c1tgtqN2pX+qcpCTLuLqgkbprGr+bIQV1gVBHwZgZfbKoupkkcMCheIcS29Lz52p8f9C3HPAvDTxcH/AoQnuJMYaAAA"""
WAVE12_PATCH_GZIP_B64 = """..."""

HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
# The retained Wave 12 combined arm, measured 2026-08-26 over four Williams
# repeats. Wave 13A changes no kernel, so this run has to land on it.
# Wave 13A, same protocol, 2026-08-27. Session drift makes this context, not
# a comparison: the same binary moved 7.4% between two sessions.
HISTORICAL_WAVE13A_TPS = 8665.4
# Isolated GEMM screening, medians of three runs on the same T4: three launches
# 123.54 us against one stacked launch 58.00 us. That bounds the ceiling; qkv is
# 8.6% of prefill, so the end-to-end expectation is roughly +4 to +6%.
SCREENED_ISOLATED_SPEEDUP = 2.130
# Pre-registered decision rule. The standard bar is the repo's retention gate;
# the second threshold is what Wave 13A measured as noise on a true no-op
# (worst paired deviation 0.61% over four interleaved repeats), so 2% is
# outside it by a wide margin. Both verdicts are reported; neither is moved
# after seeing the numbers.
RETAIN_MEDIAN = 0.05
MEASURABLE_MEDIAN = 0.02
HISTORICAL_WAVE12_COMBINED_TPS = 8096.2
# A no-op refactor is judged by how little it moves, not by how much.
NOOP_MEDIAN_BAND = 0.03
NOOP_PAIRED_FLOOR = -0.05
# Exact, chunking-invariant telemetry the instrumented arm must report for
# the pinned 244-token prompt over 24 layers. Derived from shapes, so a
# mismatch is a bug in the accounting and not a hardware detail.
EXPECT_ATTENTION_MACS = 1_285_509_120
EXPECT_ATTENTION_READ_BYTES = 755_564_544
EXPECT_ELEMENTWISE_READ_BYTES = 332_894_208
TARGET_PREFILL_TPS = 15000.0
HISTORICAL_WAVE11_TPS = 7397.5
PRODUCTION_REPEATS = 4
COLD_ITERS = 3
WARMUP_ITERS = 3
MEASURE_ITERS = 10
# Wave 13A has no kernel to profile: every PTX byte is asserted identical.
RUN_NCU = False

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave13b-{RUN_ID}"
META_REPO = ROOT / "meta"
RESULTS = ROOT / "results"
ARM_DIRS = {"wave13": ROOT / "wave13", "wave13b": ROOT / "wave13b"}
ARM_TARGETS = {name: ROOT / f"target-{name}" for name in ARM_DIRS}
FINAL_ZIP = WORK / "glcuda_t4_ceiling_wave13b_fetch_results.zip"
ROOT.mkdir(parents=True, exist_ok=False)
RESULTS.mkdir(parents=True)

def run(cmd, cwd=None, env=None, timeout=1800, check=True):
    merged = os.environ.copy()
    # A reused Kaggle kernel may carry flags from an earlier experiment. Each
    # process starts from a controlled dispatch environment.
    for key in [
        "GLCUDA_FORCE_Q8", "GLCUDA_GRID2D", "GLCUDA_R256", "GLCUDA_NO_MMA",
        "GLCUDA_FUSE_Q8_GLUE", "GLCUDA_GQA_GROUP", "GLCUDA_NTILE128",
        "GLCUDA_BSTAGE", "GLCUDA_MULTI_STREAM_PREFILL", "GLCUDA_PROFILE_PREFILL",
        "GLCUDA_PROFILE_DECODE", "GLCUDA_TELEMETRY", "GLCUDA_CACHE", "GLCUDA_JIT_VERBOSE",
    ]:
        merged.pop(key, None)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
    )
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    return p

def save_log(name, proc):
    (RESULTS / name).write_text(
        f"returncode={proc.returncode}\n\nSTDOUT\n{proc.stdout}\n\nSTDERR\n{proc.stderr}",
        encoding="utf-8",
    )

def ensure_cargo():
    cargo_home = WORK / ".wave13b-cargo"
    rustup_home = WORK / ".wave13b-rustup"
    candidates = [
        shutil.which("cargo"), cargo_home / "bin/cargo", Path.home() / ".cargo/bin/cargo",
        Path("/usr/local/cargo/bin/cargo"),
    ]
    cargo = next((Path(x) for x in candidates if x and Path(x).is_file()), None)
    if cargo is None:
        installer = ROOT / "rustup-init"
        url = "https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init"
        last_error = None
        for attempt in range(1, 6):
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "GwenLand-glcuda-Wave12/1.0", "Accept-Encoding": "identity"},
                )
                with urllib.request.urlopen(request, timeout=120) as response:
                    payload = response.read()
                if len(payload) < (1 << 20):
                    raise RuntimeError(f"rustup-init download is unexpectedly small: {len(payload)} bytes")
                installer.write_bytes(payload)
                installer.chmod(0o755)
                break
            except Exception as exc:
                last_error = exc
                if attempt == 5:
                    raise RuntimeError(f"cannot download rustup-init: {last_error}") from exc
                time.sleep(min(30, 2 ** attempt))
        install_env = {
            "CARGO_HOME": str(cargo_home), "RUSTUP_HOME": str(rustup_home),
            "PATH": f"{cargo_home / 'bin'}:{os.environ.get('PATH', '')}",
        }
        install = run(
            [installer, "-y", "--profile", "minimal", "--default-toolchain", "stable", "--no-modify-path"],
            env=install_env, timeout=1800,
        )
        save_log("rustup-init.log", install)
        cargo = cargo_home / "bin/cargo"
    cargo_bin = cargo.parent
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if cargo_home in cargo.parents:
        os.environ["CARGO_HOME"] = str(cargo_home)
        os.environ["RUSTUP_HOME"] = str(rustup_home)
    cargo = Path(shutil.which("cargo") or cargo)
    rustc = shutil.which("rustc")
    if not cargo.is_file() or not rustc:
        raise RuntimeError(f"Rust toolchain bootstrap incomplete: cargo={cargo}, rustc={rustc}")
    cargo_version = run([cargo, "--version"])
    rustc_version = run([rustc, "--version", "--verbose"])
    save_log("rust-toolchain.log", cargo_version)
    with (RESULTS / "rust-toolchain.log").open("a", encoding="utf-8") as f:
        f.write(f"\n\nRUSTC\n{rustc_version.stdout}\n{rustc_version.stderr}")
    print(f"Rust toolchain: {cargo_version.stdout.strip()} / {rustc_version.stdout.splitlines()[0]}")
    return str(cargo)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def partial_archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    return FINAL_ZIP

def fail_phase(phase):
    text = traceback.format_exc()
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": text}, indent=2), encoding="utf-8"
    )
    partial_archive()
    raise RuntimeError(f"Wave 13B {phase} failed; partial archive: {FINAL_ZIP}\n{text}")

def decode_patch(encoded, expected, label):
    data = gzip.decompress(base64.b64decode(encoded))
    got = hashlib.sha256(data).hexdigest()
    if got != expected:
        raise RuntimeError(f"{label} patch hash mismatch: {got} != {expected}")
    path = RESULTS / f"{label}.patch"
    path.write_bytes(data)
    return path

def ptx_entry(text, name):
    start = text.find(f".visible .entry {name}(")
    if start < 0:
        raise RuntimeError(f"PTX entry missing: {name}")
    brace = text.find("{", start)
    depth = 0
    for index in range(brace, len(text)):
        if text[index] == "{":
            depth += 1
        elif text[index] == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1]
    raise RuntimeError(f"unbalanced PTX entry: {name}")

assert callable(time.monotonic)
PATCHES = {
    "wave3": decode_patch(WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256, "wave3"),
    "wave4": decode_patch(WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256, "wave4"),
    "wave11": decode_patch(WAVE11_PATCH_GZIP_B64, WAVE11_PATCH_SHA256, "wave11"),
    "wave12": decode_patch(WAVE12_PATCH_GZIP_B64, WAVE12_PATCH_SHA256, "wave12"),
    "wave13": decode_patch(WAVE13_PATCH_GZIP_B64, WAVE13_PATCH_SHA256, "wave13"),
    "wave13b": decode_patch(WAVE13B_PATCH_GZIP_B64, WAVE13B_PATCH_SHA256, "wave13b"),
}

try:
    gpu = run(
        ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"],
        timeout=60,
    )
    rows = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
    if not rows:
        raise RuntimeError("nvidia-smi returned no GPU")
    fields = [x.strip() for x in rows[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 13B requires Tesla T4 sm_75, got {rows[0]}")
    GPU_INFO = {
        "raw": rows[0], "index": fields[0], "name": fields[1],
        "compute_cap": fields[2], "memory_mib": fields[3], "driver": fields[4],
    }
    (RESULTS / "gpu.json").write_text(json.dumps(GPU_INFO, indent=2), encoding="utf-8")
    CARGO = ensure_cargo()

    clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
    save_log("git-clone.log", clone)
    obj = run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO, check=False)
    if obj.returncode:
        fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
        save_log("git-fetch-base.log", fetch)
        obj = run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO)
    if obj.stdout.strip() != "commit":
        raise RuntimeError(f"BASE_REV is not a commit: {obj.stdout!r}")

    for name, path in ARM_DIRS.items():
        run(["git", "worktree", "add", "--detach", path, BASE_REV], cwd=META_REPO)
        # Wave 12 was retained (bstage +9.85%, combined +12.95%), so it is
        # the baseline now and BOTH arms carry it. Only Wave 13A separates them.
        # Wave 13A was retained as a measured no-op that lit up two dark
        # stages, so it is the baseline now and both arms carry it.
        stack = [
            PATCHES["wave3"], PATCHES["wave4"], PATCHES["wave11"],
            PATCHES["wave12"], PATCHES["wave13"],
        ]
        if name == "wave13b":
            stack.append(PATCHES["wave13b"])
        for patch in stack:
            run(["git", "apply", "--whitespace=error", patch], cwd=path)
        run(["git", "diff", "--check"], cwd=path)

    # A gate that reads prose is a gate that lies (Wave 12 asserted "cp.async"
    # was absent and matched a comment saying sm_75 has none), so every source
    # assertion below runs on code only.
    def rust_code_only(text):
        return re.sub(r"//.*", "", text)

    def read(arm, rel):
        return (ARM_DIRS[arm] / rel).read_text(encoding="utf-8")

    def ptx_entries(text):
        """Every `.visible .entry` block, keyed by name."""
        out = {}
        for match in re.finditer(r"\.visible \.entry (\w+)\(", text):
            name = match.group(1)
            brace = text.index("{", match.start())
            depth = 0
            for index in range(brace, len(text)):
                if text[index] == "{":
                    depth += 1
                elif text[index] == "}":
                    depth -= 1
                    if depth == 0:
                        out[name] = text[match.start():index + 1]
                        break
        return out

    base_main = ptx_entries(read("wave13", "glcuda/src/kernels/glcuda.ptx"))
    cand_main = ptx_entries(read("wave13b", "glcuda/src/kernels/glcuda.ptx"))
    base_sm75 = (ARM_DIRS["wave13"] / "glcuda/src/kernels/glcuda_sm75.ptx").read_bytes()
    cand_sm75 = (ARM_DIRS["wave13b"] / "glcuda/src/kernels/glcuda_sm75.ptx").read_bytes()
    changed = sorted(n for n in base_main if base_main[n] != cand_main.get(n))
    # Exactly the kernels that had to learn a row stride, and nothing else. A
    # sixth name appearing here means the patch touched a kernel nobody
    # reviewed; a missing name means the migration is incomplete.
    EXPECTED_CHANGED = [
        "gl_add_bias_rows_f32",
        "gl_attn_decode_rows_f32",
        "gl_attn_decode_rows_gqa7_f32",
        "gl_attn_rows_probe",
        "gl_kv_write_rows",
        "gl_rope_rows_f32",
    ]
    model_rs = rust_code_only(read("wave13b", "glcuda/src/model.rs"))
    runner_rs = rust_code_only(read("wave13b", "glcuda/src/runner.rs"))
    base_model_rs = rust_code_only(read("wave13", "glcuda/src/model.rs"))
    base_runner_rs = rust_code_only(read("wave13", "glcuda/src/runner.rs"))
    STRUCTURAL = {
        # The MMA GEMM path is untouched: Wave 13B changes which rows one launch
        # covers, never how a row is computed.
        "sm75_ptx_byte_identical": base_sm75 == cand_sm75,
        "entry_names_unchanged": sorted(base_main) == sorted(cand_main),
        "changed_entries_are_exactly_the_stride_migration": changed == EXPECTED_CHANGED,
        "stride_params_added": all(
            "p_row_stride" in cand_main[n] or "p_src_stride" in cand_main[n]
            or "p_q_row_stride" in cand_main[n]
            for n in EXPECTED_CHANGED
        ),
        "baseline_had_no_stride_params": not any(
            "p_row_stride" in b or "p_src_stride" in b or "p_q_row_stride" in b
            for b in base_main.values()
        ),
        # One contiguous upload, three views, and a stacked matrix to launch.
        "contiguous_upload_present": "fn up_qkv(" in model_rs,
        "stacked_matrix_field": "w_qkv: Option<GpuMat>" in model_rs,
        "baseline_has_neither": "up_qkv" not in base_model_rs and "w_qkv" not in base_model_rs,
        # Guarded dispatch: models with per-head q/k norms keep the three-launch
        # path, because rms_norm_rows walks contiguous head rows and a stacked
        # slab makes that a two-level mapping.
        "guarded_on_head_norms": "qkv_stacked" in runner_rs and "q_norm.is_none()" in runner_rs,
        "baseline_launches_three": "&layer.wq," in base_runner_rs,
    }
    if not all(STRUCTURAL.values()):
        raise RuntimeError(
            f"Wave 13B structural contract failed: {STRUCTURAL} changed={changed}"
        )
    STRUCTURAL["changed_entries"] = changed
    (RESULTS / "structural.json").write_text(json.dumps(STRUCTURAL, indent=2), encoding="utf-8")
    STACK_OK = True
    print(f"Wave 13B stack ready at {ROOT}")
    print(json.dumps({"gpu": GPU_INFO, "structural": STRUCTURAL}, indent=2))
except Exception:
    fail_phase("bootstrap")


## 2 - Resumable pinned model fetch

In [ ]:
if not globals().get("STACK_OK"):
    raise RuntimeError("Bootstrap gate did not pass")

def fetch_pinned_model():
    model = WORK / HF_FILENAME
    part = WORK / f"{HF_FILENAME}.part"
    if model.is_file() and model.stat().st_size == HF_EXPECTED_BYTES and sha256_file(model) == HF_EXPECTED_SHA256:
        return model
    if model.exists():
        model.unlink()
    if part.exists():
        part_size = part.stat().st_size
        if part_size == HF_EXPECTED_BYTES:
            if sha256_file(part) == HF_EXPECTED_SHA256:
                part.replace(model)
                return model
            part.unlink()
        elif part_size > HF_EXPECTED_BYTES:
            part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave12/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent": "GwenLand-glcuda-Wave12/1.0", "Accept-Encoding": "identity"}),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(f"model fetch {downloaded / (1 << 20):.1f}/{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB")
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            digest = sha256_file(part)
            if digest != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {digest}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}")
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))

try:
    MODEL_PATH = fetch_pinned_model()
    MODEL_META = {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": MODEL_PATH.stat().st_size, "sha256": sha256_file(MODEL_PATH)}
    (RESULTS / "model.json").write_text(json.dumps(MODEL_META, indent=2), encoding="utf-8")
    MODEL_OK = True
    print(json.dumps(MODEL_META, indent=2))
except Exception:
    fail_phase("model-fetch")

## 3 - Build, ptxas, and the cost of a row stride

In [ ]:
if not globals().get("MODEL_OK"):
    raise RuntimeError("Model gate did not pass")

def parse_ptxas(text):
    resources = {}
    current = None
    for line in text.splitlines():
        m = re.search(r"Function properties for\s+'?([^'\s]+)'?", line)
        if m:
            current = m.group(1)
            resources.setdefault(current, {"spill_stores": 0, "spill_loads": 0})
        if current:
            m = re.search(r"(\d+) bytes spill stores, (\d+) bytes spill loads", line)
            if m:
                resources[current]["spill_stores"] = int(m.group(1))
                resources[current]["spill_loads"] = int(m.group(2))
            # Read the register count and the smem count with separate regexes:
            # ptxas puts a varying number of fields between them, and one regex
            # spanning both reported smem 0 for every kernel in Wave 12.
            m = re.search(r"Used\s+(\d+) registers", line)
            if m:
                resources[current]["registers"] = int(m.group(1))
                smem = re.search(r"(\d+) bytes smem", line)
                resources[current]["smem_bytes"] = int(smem.group(1)) if smem else 0
    return resources

try:
    PTXAS = shutil.which("ptxas")
    if not PTXAS:
        raise RuntimeError("ptxas not found in Kaggle CUDA image")
    BUILD_INFO = {}
    TESTS = {}
    for name, path in ARM_DIRS.items():
        target = ARM_TARGETS[name]
        env = {"CARGO_TARGET_DIR": str(target)}
        check = run([CARGO, "check", "-p", "glcuda", "--locked"], cwd=path, env=env, timeout=3600)
        save_log(f"cargo-check-{name}.log", check)
        tests = run([CARGO, "test", "-p", "glcuda", "--lib", "--locked"], cwd=path, env=env, timeout=3600)
        save_log(f"cargo-lib-{name}.log", tests)
        hay = tests.stdout + "\n" + tests.stderr
        m = re.search(r"test result: ok\. (\d+) passed", hay)
        if not m:
            raise RuntimeError(f"lib tests did not report a passing count for {name}")
        TESTS[name] = {"lib_passed": int(m.group(1))}
        build_glbench = run([CARGO, "build", "--release", "-p", "glbench", "--locked"], cwd=path, env=env, timeout=7200)
        save_log(f"cargo-build-{name}.log", build_glbench)
        BUILD_INFO[name] = {"glbench": str(target / "release/glbench")}

    PTXAS_RESOURCES = {}
    for name, path in ARM_DIRS.items():
        for module in ["glcuda.ptx", "glcuda_sm75.ptx"]:
            src = path / "glcuda/src/kernels" / module
            out = RESULTS / f"{name}-{module}.cubin"
            p = run([PTXAS, "-arch=sm_75", "-v", "--warn-on-spills", src, "-o", out], timeout=1800, check=False)
            save_log(f"ptxas-{name}-{module}.log", p)
            if p.returncode:
                raise RuntimeError(f"ptxas failed for {name}/{module}")
            parsed = parse_ptxas(p.stdout + "\n" + p.stderr)
            PTXAS_RESOURCES[f"{name}/{module}"] = parsed
            if any(v.get("spill_stores", 0) or v.get("spill_loads", 0) for v in parsed.values()):
                raise RuntimeError(f"spill detected in {name}/{module}: {parsed}")

    # The tensor-core module is byte-identical, so its resources must be too.
    # That is the check that the GEMM this wave re-aims was not itself changed.
    if PTXAS_RESOURCES["wave13/glcuda_sm75.ptx"] != PTXAS_RESOURCES["wave13b/glcuda_sm75.ptx"]:
        raise RuntimeError(f"MMA kernel resources moved: {PTXAS_RESOURCES}")
    for kernel in ["gl_gemm_mma_q8", "gl_gemm_mma_q8_bstage"]:
        if not PTXAS_RESOURCES["wave13b/glcuda_sm75.ptx"].get(kernel, {}).get("smem_bytes"):
            raise RuntimeError(f"ptxas resource line did not parse for {kernel}")

    # A row stride costs a register or two in the kernels that gained it. Report
    # the delta rather than asserting a number nobody measured.
    STRIDE_COST = {}
    old = PTXAS_RESOURCES["wave13/glcuda.ptx"]
    new = PTXAS_RESOURCES["wave13b/glcuda.ptx"]
    for kernel in sorted(set(old) & set(new)):
        if old[kernel] != new[kernel]:
            STRIDE_COST[kernel] = {"before": old[kernel], "after": new[kernel]}

    (RESULTS / "ptxas-resources.json").write_text(json.dumps(PTXAS_RESOURCES, indent=2), encoding="utf-8")
    (RESULTS / "stride-cost.json").write_text(json.dumps(STRIDE_COST, indent=2), encoding="utf-8")
    (RESULTS / "tests.json").write_text(json.dumps(TESTS, indent=2), encoding="utf-8")
    BUILD_OK = True
    print(json.dumps({"tests": TESTS, "stride_cost": STRIDE_COST}, indent=2))
except Exception:
    fail_phase("build-ptxas")


## 4 - Hardware correctness: strided must equal packed

In [ ]:
if not globals().get("BUILD_OK"):
    raise RuntimeError("Build gate did not pass")
try:
    PARITY = {}
    for name, path in ARM_DIRS.items():
        env = {
            "CARGO_TARGET_DIR": str(ARM_TARGETS[name]),
            "CUDA_VISIBLE_DEVICES": "0",
            "GLCUDA_GQA_GROUP": "1", "GLCUDA_FUSE_Q8_GLUE": "1",
            "GLCUDA_GRID2D": "1", "GLCUDA_NTILE128": "1", "GLCUDA_BSTAGE": "1",
        }
        p = run(
            [CARGO, "test", "--release", "-p", "glcuda", "--test", "parity", "--locked", "--", "--test-threads=1", "--nocapture"],
            cwd=path, env=env, timeout=7200, check=False,
        )
        save_log(f"hardware-parity-{name}.log", p)
        hay = p.stdout + "\n" + p.stderr
        # A skipped parity test passes, and every exactness claim here rests on
        # these actually executing on the T4.
        if p.returncode or "SKIP:" in hay:
            raise RuntimeError(f"hardware parity failed or skipped for {name}")
        m = re.search(r"test result: ok\. (\d+) passed", hay)
        PARITY[name] = {"status": "PASS", "tests": int(m.group(1)) if m else None}
        strided = "strided_projection_slices_match_the_packed_layout_through_attention" in hay
        if (name == "wave13b") != strided:
            raise RuntimeError(f"strided-vs-packed test presence wrong for {name}: {strided}")
    # The new test is the one that proves the whole stride migration: the same
    # bias/RoPE/KV-write/attention chain over a column slice of a stacked slab
    # must reproduce the packed chain bit for bit.
    if PARITY["wave13b"]["tests"] <= PARITY["wave13"]["tests"]:
        raise RuntimeError(f"Wave 13B added no parity coverage: {PARITY}")
    (RESULTS / "parity.json").write_text(json.dumps(PARITY, indent=2), encoding="utf-8")
    CORRECTNESS_OK = True
    print(json.dumps(PARITY, indent=2))
except Exception:
    fail_phase("hardware-correctness")


## 5 - Interleaved production A/B and stage attribution

In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise RuntimeError("Hardware correctness gate did not pass")

prompt_unit = "Measure this deterministic systems prompt carefully. Explain how token-parallel integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
FIXED_PROMPT = prompt_unit * 8
COMMON_ENV = {
    "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
    "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_GQA_GROUP": "1",
    "GLCUDA_NTILE128": "1", "GLCUDA_BSTAGE": "1",
}
EXPECTED_CONTRACT = {
    "exact_fusion": True, "gqa_group": True, "grid2d": True,
    "r256": False, "ntile128": True, "bstage": True,
}
ARMS = ["wave13", "wave13b"]
ORDERS = [ARMS, ARMS[::-1], ARMS, ARMS[::-1]]
assert len(ORDERS) == PRODUCTION_REPEATS
assert all(len({o[p] for o in ORDERS}) == 2 for p in range(2))

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)

def contract_from_log(hay):
    matches = re.findall(r"\[glcuda-contract\]\s*(\{[^\n]+\})", hay)
    if not matches:
        raise RuntimeError("machine-readable glcuda contract missing")
    contract = json.loads(matches[-1])
    contract.setdefault("ntile128", False)
    contract.setdefault("bstage", False)
    return contract

def qkv_dispatch_from_log(hay):
    """Which QKV path the binary actually took, straight from the engine."""
    matches = re.findall(r"\[glcuda-qkv\]\s*(\{[^\n]+\})", hay)
    return json.loads(matches[-1]) if matches else None

def session_stats(path, expected_iters=MEASURE_ITERS):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    engine = data.get("engine") or {}
    workload = data.get("workload") or {}
    if engine.get("name") != "glcuda" or engine.get("backend") != "cuda" or not engine.get("available"):
        raise RuntimeError(f"wrong engine contract: {engine}")
    expected_workload = {
        "engine": "glcuda", "kind": "prefill", "prompt": FIXED_PROMPT,
        "seed": 42, "temperature": 0.0, "max_new_tokens": 1,
        "cold_iters": COLD_ITERS, "warmup_iters": WARMUP_ITERS,
        "measure_iters": expected_iters, "verify_against": "glproc",
    }
    for key, value in expected_workload.items():
        if workload.get(key) != value:
            raise RuntimeError(f"workload {key} mismatch: {workload.get(key)!r} != {value!r}")
    validation = data.get("validation") or {}
    if validation.get("passed") is not True:
        raise RuntimeError(f"session validation failed: {validation}")
    parity = [f for f in validation.get("findings", []) if f.get("check") == "parity"]
    if not parity or parity[-1].get("severity") != "info":
        raise RuntimeError(f"oracle parity evidence missing: {validation}")
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", ""))
    if not match or match.group(1) != match.group(2) or int(match.group(2)) != 50:
        raise RuntimeError(f"exact 50/50 oracle failed: {parity[-1] if parity else None}")
    iterations = (data.get("measurements") or {}).get("iterations") or []
    if len(iterations) != expected_iters:
        raise RuntimeError(f"wrong measured iteration count: {len(iterations)}")
    prompt_counts = [int(x.get("prompt_tokens", 0)) for x in iterations]
    prefill_ms = [float(x.get("prefill_ms", 0)) for x in iterations]
    decode_ms = [float(x.get("decode_ms", 0)) for x in iterations]
    if len(set(prompt_counts)) != 1 or prompt_counts[0] != 244 or any(x <= 0 for x in prefill_ms + decode_ms):
        raise RuntimeError(f"malformed iteration payload: {iterations}")
    tps = [prompt_counts[0] * 1000.0 / x for x in prefill_ms]
    return {
        "prompt_tokens": prompt_counts[0], "prefill_p50": percentile(tps, .5),
        "prefill_mean": statistics.mean(tps), "prefill_p95": percentile(tps, .95),
        "latency_p95_ms": percentile(prefill_ms, .95), "latency_p99_ms": percentile(prefill_ms, .99),
        "decode_p50": percentile([1000.0 / x for x in decode_ms], .5), "oracle": "50/50",
    }

def gpu_snapshot(repeat, position, arm, phase):
    query = "timestamp,index,name,pstate,temperature.gpu,power.draw,clocks.current.sm,clocks.current.memory,utilization.gpu,memory.used"
    p = run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits", "-i", "0"], timeout=60, check=False)
    return {"repeat": repeat, "position": position, "arm": arm, "phase": phase, "returncode": p.returncode, "csv": p.stdout.strip()}

def glbench_cmd(out, cold, warmup, iters):
    return ["run", "--engine", "glcuda", "--model", MODEL_PATH, "--prompt", FIXED_PROMPT, "--tokens", "1", "--cold-iters", str(cold), "--warmup", str(warmup), "--iters", str(iters), "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", out]

def run_arm(arm, out, cold, warmup, iters, extra_env=None):
    env = {"CUDA_VISIBLE_DEVICES": "0", **COMMON_ENV, **(extra_env or {})}
    return run([BUILD_INFO[arm]["glbench"], *glbench_cmd(out, cold, warmup, iters)], cwd=ARM_DIRS[arm], env=env, timeout=14400, check=False)

def check_arm_log(arm, hay):
    got = contract_from_log(hay)
    if {k: got.get(k) for k in EXPECTED_CONTRACT} != EXPECTED_CONTRACT:
        raise RuntimeError(f"dispatch truth-table mismatch for {arm}: {got}")
    if "[glcuda] GLCUDA_FORCE_Q8:" not in hay:
        raise RuntimeError(f"forced-Q8 loader contract missing for {arm}")
    # The engine states which QKV path it took, so the arm is auditable rather
    # than inferred from its timing.
    qkv = qkv_dispatch_from_log(hay)
    if arm == "wave13b":
        if not qkv or qkv["stacked_layers"] != qkv["layers"] or qkv["rows"] != 1152:
            raise RuntimeError(f"candidate did not take the stacked QKV path: {qkv}")
    elif qkv is not None:
        raise RuntimeError(f"baseline reported a QKV stack it cannot have: {qkv}")
    return qkv

try:
    QKV_DISPATCH = {}
    for arm in ARMS:
        out = RESULTS / f"stabilize-{arm}.json"
        p = run_arm(arm, out, 0, 1, 1)
        save_log(f"stabilize-{arm}.log", p)
        if p.returncode:
            raise RuntimeError(f"stabilization failed: {arm}")
        QKV_DISPATCH[arm] = check_arm_log(arm, p.stdout + "\n" + p.stderr)

    PROD_RECORDS = []
    HARDWARE_SNAPSHOTS = []
    for repeat, order in enumerate(ORDERS):
        for position, arm in enumerate(order):
            archive = RESULTS / f"glbench-{repeat}-{position}-{arm}.json"
            HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, arm, "before"))
            p = run_arm(arm, archive, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)
            HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, arm, "after"))
            save_log(f"glbench-{repeat}-{position}-{arm}.log", p)
            if p.returncode:
                raise RuntimeError(f"glbench failed: {arm} repeat {repeat}")
            check_arm_log(arm, p.stdout + "\n" + p.stderr)
            stats = session_stats(archive)
            PROD_RECORDS.append({"repeat": repeat, "position": position, "arm": arm, **stats})
            print(f"{arm:9s} r{repeat} p{position}: {stats['prefill_p50']:.1f} tok/s | P95 {stats['latency_p95_ms']:.2f} ms | oracle {stats['oracle']}")

    PROD_SUMMARY = []
    for arm in ARMS:
        rows = [x for x in PROD_RECORDS if x["arm"] == arm]
        PROD_SUMMARY.append({
            "arm": arm,
            "session_p50_median": statistics.median(x["prefill_p50"] for x in rows),
            "session_mean_median": statistics.median(x["prefill_mean"] for x in rows),
            "latency_p95_median_ms": statistics.median(x["latency_p95_ms"] for x in rows),
            "latency_p99_median_ms": statistics.median(x["latency_p99_ms"] for x in rows),
            "decode_p50_median": statistics.median(x["decode_p50"] for x in rows),
            "sessions": len(rows), "positions": [x["position"] for x in rows],
        })
    summary = {x["arm"]: x for x in PROD_SUMMARY}
    paired = []
    for repeat in range(PRODUCTION_REPEATS):
        base = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == "wave13")
        cand = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == "wave13b")
        paired.append({
            "repeat": repeat,
            "prefill_p50_delta": cand["prefill_p50"] / base["prefill_p50"] - 1,
            "prefill_mean_delta": cand["prefill_mean"] / base["prefill_mean"] - 1,
            "p95_latency_delta": cand["latency_p95_ms"] / base["latency_p95_ms"] - 1,
            "p99_latency_delta": cand["latency_p99_ms"] / base["latency_p99_ms"] - 1,
            "decode_p50_delta": cand["decode_p50"] / base["decode_p50"] - 1,
        })
    median_delta = summary["wave13b"]["session_p50_median"] / summary["wave13"]["session_p50_median"] - 1
    DECISION = {
        "median_delta": median_delta,
        "paired": paired,
        "retain_bar": RETAIN_MEDIAN,
        "measurable_bar": MEASURABLE_MEDIAN,
        "all_paired_positive": all(x["prefill_p50_delta"] > 0 for x in paired),
        "all_paired_over_retain_bar": all(x["prefill_p50_delta"] >= RETAIN_MEDIAN for x in paired),
        "all_paired_over_measurable_bar": all(x["prefill_p50_delta"] >= MEASURABLE_MEDIAN for x in paired),
        "tail_pass": all(x["p95_latency_delta"] <= .05 and x["p99_latency_delta"] <= .05 for x in paired),
        "decode_pass": all(x["decode_p50_delta"] >= -.05 for x in paired),
        "oracle_pass": all(x["oracle"] == "50/50" for x in PROD_RECORDS),
    }
    gates = DECISION["tail_pass"] and DECISION["decode_pass"] and DECISION["oracle_pass"]
    if gates and median_delta >= RETAIN_MEDIAN and DECISION["all_paired_over_retain_bar"]:
        DECISION["verdict"] = "RETAIN (clears the standard 5% bar)"
    elif gates and median_delta >= MEASURABLE_MEDIAN and DECISION["all_paired_over_measurable_bar"]:
        DECISION["verdict"] = "MEASURABLE WIN below the 5% bar - JinXSuper's call"
    elif gates and median_delta > 0:
        DECISION["verdict"] = "POSITIVE BUT INSIDE NOISE"
    else:
        DECISION["verdict"] = "REJECT"

    # ---- telemetry, interleaved -----------------------------------------
    TELEMETRY = {}
    for repeat in range(2):
        for arm in (ARMS if repeat % 2 == 0 else ARMS[::-1]):
            out = RESULTS / f"telemetry-{repeat}-{arm}.json"
            p = run_arm(arm, out, 0, 3, 1, {"GLCUDA_TELEMETRY": "1"})
            save_log(f"telemetry-{repeat}-{arm}.log", p)
            if p.returncode:
                raise RuntimeError(f"telemetry failed: {arm}")
            data = json.loads(out.read_text(encoding="utf-8"))
            stages = (((data.get("telemetry") or {}).get("prefill") or {}).get("stages") or [])
            if not stages:
                raise RuntimeError(f"telemetry emitted no stages: {arm}")
            TELEMETRY.setdefault(arm, []).append({x["name"]: x for x in stages})

    def stage_stat(arm, stage):
        rows = []
        for sample in TELEMETRY[arm]:
            entry = sample[stage]
            ms = float(entry.get("total_ms") or 0)
            macs = int(entry.get("macs") or 0)
            byts = int(entry.get("bytes_read") or 0)
            total = sum(float(v.get("total_ms") or 0) for v in sample.values())
            rows.append({
                "ms": ms, "macs": macs, "bytes": byts,
                "share": ms / total if total else None,
                "gmac_per_s": (macs / (ms / 1000) / 1e9) if ms and macs else None,
            })
        return {
            "ms_median": statistics.median(r["ms"] for r in rows),
            "share_median": statistics.median(r["share"] for r in rows),
            "gmac_per_s": statistics.median([r["gmac_per_s"] for r in rows]) if rows[0]["gmac_per_s"] else None,
            "macs": rows[0]["macs"], "bytes": rows[0]["bytes"],
        }

    STAGES = {}
    for stage in ["qkv", "attention", "ffn_gate_up", "ffn_down", "ffn_elementwise", "attn_out"]:
        STAGES[stage] = {arm: stage_stat(arm, stage) for arm in ARMS}
    # Wave 13A's constants must not have moved: this wave changes launch
    # geometry, not arithmetic, so the accounting is a regression check.
    for arm in ARMS:
        if STAGES["attention"][arm]["macs"] != EXPECT_ATTENTION_MACS:
            raise RuntimeError(f"attention MACs moved for {arm}: {STAGES['attention'][arm]}")
        if STAGES["attention"][arm]["bytes"] != EXPECT_ATTENTION_READ_BYTES:
            raise RuntimeError(f"attention bytes moved for {arm}: {STAGES['attention'][arm]}")
    # Same arithmetic, same weight traffic, one launch instead of three.
    if STAGES["qkv"]["wave13"]["macs"] != STAGES["qkv"]["wave13b"]["macs"]:
        raise RuntimeError(f"qkv MACs differ between arms: {STAGES['qkv']}")
    QKV_EFFECT = {
        "gmac_per_s_before": STAGES["qkv"]["wave13"]["gmac_per_s"],
        "gmac_per_s_after": STAGES["qkv"]["wave13b"]["gmac_per_s"],
        "share_before": STAGES["qkv"]["wave13"]["share_median"],
        "share_after": STAGES["qkv"]["wave13b"]["share_median"],
        "isolated_screening_speedup": SCREENED_ISOLATED_SPEEDUP,
    }
    if QKV_EFFECT["gmac_per_s_before"]:
        QKV_EFFECT["stage_speedup"] = QKV_EFFECT["gmac_per_s_after"] / QKV_EFFECT["gmac_per_s_before"]

    for name, payload in [
        ("hardware-snapshots", HARDWARE_SNAPSHOTS), ("production-records", PROD_RECORDS),
        ("production-summary", PROD_SUMMARY), ("decision", DECISION),
        ("stages", STAGES), ("qkv-effect", QKV_EFFECT), ("qkv-dispatch", QKV_DISPATCH),
    ]:
        (RESULTS / f"{name}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    PROD_OK = True
    print(json.dumps(PROD_SUMMARY, indent=2))
    print(json.dumps(DECISION, indent=2))
    print(json.dumps(QKV_EFFECT, indent=2))
except Exception:
    fail_phase("production-ab")


## 6 - Report and evidence archive

In [ ]:
if not globals().get("PROD_OK"):
    raise RuntimeError("A/B gate did not complete")
try:
    summary = {x["arm"]: x for x in PROD_SUMMARY}
    lines = [
        "# glcuda T4 Ceiling - Wave 13B (stacked QKV projection)", "",
        f"- notebook: {NOTEBOOK_BUILD}",
        f"- GPU: {GPU_INFO['raw']}",
        f"- baseline revision: {BASE_REV}",
        f"- Wave 13A patch (now baseline): {WAVE13_PATCH_SHA256}",
        f"- Wave 13B patch: {WAVE13B_PATCH_SHA256}",
        f"- model SHA-256: {MODEL_META['sha256']}",
        f"- lib tests: {TESTS['wave13']['lib_passed']} -> {TESTS['wave13b']['lib_passed']}",
        f"- parity tests: {PARITY['wave13']['tests']} -> {PARITY['wave13b']['tests']} (both RAN on the T4)",
        f"- QKV dispatch: {json.dumps(QKV_DISPATCH)}",
        "",
        "## What changed",
        "",
        "- `W_q`, `W_k` and `W_v` upload as one contiguous image; the three",
        "  `GpuMat`s are views onto it, so the stack costs no extra VRAM and",
        "  decode keeps using the views unchanged.",
        "- Prefill runs ONE GEMM over all q_dim + 2*kv_dim rows. Q, K and V are",
        "  column slices of one slab, which is why every consumer now takes a",
        "  row stride instead of deriving it from its own row width.",
        "- The tensor-core module is byte-identical: this wave changes which",
        "  rows a launch covers, never how a row is computed.",
        f"- PTX entries changed: {', '.join(STRUCTURAL['changed_entries'])}",
        "",
        "## Interleaved, position-balanced A/B",
        "",
        "| arm | P50 tok/s | mean | P95 latency ms | decode P50 | sessions |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for row in PROD_SUMMARY:
        lines.append(
            f"| {row['arm']} | {row['session_p50_median']:.1f} | {row['session_mean_median']:.1f} |"
            f" {row['latency_p95_median_ms']:.2f} | {row['decode_p50_median']:.1f} | {row['sessions']} |"
        )
    lines += [
        "",
        f"- median delta: **{DECISION['median_delta'] * 100:+.2f}%**",
        "- paired deltas: " + ", ".join(f"{x['prefill_p50_delta'] * 100:+.2f}%" for x in DECISION["paired"]),
        f"- bars: retain {RETAIN_MEDIAN * 100:.0f}%, measurable {MEASURABLE_MEDIAN * 100:.0f}%"
        f" (Wave 13A measured 0.61% as the worst paired deviation on a true no-op)",
        f"- tails within 5%: {DECISION['tail_pass']} | decode within 5%: {DECISION['decode_pass']}"
        f" | oracle 50/50 everywhere: {DECISION['oracle_pass']}",
        f"- Wave 13A in its own session: {HISTORICAL_WAVE13A_TPS:.1f} tok/s (context, not a comparison)",
        "",
        f"## Verdict: {DECISION['verdict']}",
        "",
        "## Where the change landed",
        "",
        "| stage | share before | share after | GMAC/s before | GMAC/s after |",
        "|---|---:|---:|---:|---:|",
    ]
    for stage, arms in STAGES.items():
        before, after = arms["wave13"], arms["wave13b"]
        gb = f"{before['gmac_per_s']:.0f}" if before["gmac_per_s"] else "-"
        ga = f"{after['gmac_per_s']:.0f}" if after["gmac_per_s"] else "-"
        lines.append(
            f"| {stage} | {before['share_median'] * 100:.1f}% | {after['share_median'] * 100:.1f}% | {gb} | {ga} |"
        )
    if QKV_EFFECT.get("stage_speedup"):
        lines += [
            "",
            f"- qkv stage: {QKV_EFFECT['gmac_per_s_before']:.0f} -> {QKV_EFFECT['gmac_per_s_after']:.0f} GMAC/s"
            f" (**{QKV_EFFECT['stage_speedup']:.2f}x**), against {SCREENED_ISOLATED_SPEEDUP:.2f}x for the isolated GEMM.",
            "- The stage also contains the input RMSNorm and quantize, so it can",
            "  never reach the isolated ratio; the gap between them is that glue.",
        ]
    lines += [
        "",
        "## Interpretation rule",
        "",
        "Retained when the median and every paired repeat clear 5%, tails and",
        "decode stay within 5%, the oracle is exactly 50/50 in every session,",
        "the engine reports the stacked path in every arm that claims it, and",
        "the attention accounting is unchanged. Both bars were fixed before the",
        "numbers were seen; neither moved afterwards.",
        "",
        f"Results directory: {RESULTS}",
    ]
    report_text = "\n".join(lines)
    (RESULTS / "REPORT.md").write_text(report_text, encoding="utf-8")
    index = {}
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            index[str(path.relative_to(RESULTS))] = sha256_file(path)
    (RESULTS / "artifact-index.sha256").write_text(
        "\n".join(f"{v}  {k}" for k, v in index.items()), encoding="utf-8"
    )
    partial_archive()
    print(report_text)
    print(f"\nDownload archive: {FINAL_ZIP} ({FINAL_ZIP.stat().st_size / (1 << 20):.2f} MB)")
except Exception:
    fail_phase("report")
